## Ide kezdem el a Binance letöltését

In [ ]:
from __future__ import annotations

import io
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import requests

In [ ]:
@dataclass(frozen=True)
class MarketDataRequest:
    symbol: str
    interval: str
    year: int
    month: int


@dataclass(frozen=True)
class DownloadResult:
    provider: str
    url: str
    files: list[Path]


class MarketDataProvider(Protocol):
    name: str

    def build_url(self, request: MarketDataRequest) -> str:
        ...

    def download(self, request: MarketDataRequest, output_dir: Path) -> DownloadResult:
        ...


In [ ]:
class BinanceProvider:
    name = "binance"
    base_url = "https://data.binance.vision"

    def __init__(self, timeout_sec: int = 30):
        self.timeout_sec = timeout_sec
        self.session = requests.Session()

    def build_url(self, request: MarketDataRequest) -> str:
        month = f"{request.month:02d}"

        return (
            f"{self.base_url}/data/spot/monthly/klines/"
            f"{request.symbol}/{request.interval}/"
            f"{request.symbol}-{request.interval}-{request.year}-{month}.zip"
        )

    def download(self, request: MarketDataRequest, output_dir: Path) -> DownloadResult:
        url = self.build_url(request)

        response = self.session.get(url, timeout=self.timeout_sec)

        if response.status_code == 404:
            raise FileNotFoundError(f"Binance file not found: {url}")

        if response.status_code != 200:
            raise RuntimeError(f"Binance HTTP error {response.status_code}: {url}")

        files = self._extract_csv_files(
            zip_bytes=response.content,
            output_dir=output_dir,
        )

        return DownloadResult(
            provider=self.name,
            url=url,
            files=files,
        )

    def _extract_csv_files(self, zip_bytes: bytes, output_dir: Path) -> list[Path]:
        output_dir.mkdir(parents=True, exist_ok=True)

        extracted_files: list[Path] = []

        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
            for member in archive.infolist():
                if not member.filename.lower().endswith(".csv"):
                    continue

                member_path = Path(member.filename)

                if member_path.is_absolute() or ".." in member_path.parts:
                    raise ValueError(f"Unsafe ZIP entry: {member.filename}")

                archive.extract(member, output_dir)
                extracted_files.append(output_dir / member.filename)

        if not extracted_files:
            raise ValueError("No CSV file found in Binance ZIP.")

        return extracted_files


In [ ]:
provider = BinanceProvider()

request = MarketDataRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result = provider.download(
    request=request,
    output_dir=Path("data/binance"),
)

result


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest


provider = BinanceProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result = provider.download(
    request=request,
    output_dir=Path("data/raw/binance"),
)

result


In [ ]:
from pathlib import Path
import csv
from datetime import datetime, timezone


csv_path = result.files[0]

expected_column_count = 12

row_count = 0
bad_column_count = 0
bad_timestamp_count = 0
bad_number_count = 0
duplicate_timestamp_count = 0

timestamps = set()
min_timestamp = None
max_timestamp = None
previous_timestamp = None
is_time_ordered = True

number_columns = [1, 2, 3, 4, 5, 7, 8, 9, 10]

with open(csv_path, "r", encoding="utf-8") as file:
    reader = csv.reader(file)

    for row in reader:
        row_count += 1

        if len(row) != expected_column_count:
            bad_column_count += 1
            continue

        try:
            timestamp = int(row[0])
            # timestamp_dt = datetime.fromtimestamp(timestamp / 1000)
            timestamp_dt = datetime.fromtimestamp(timestamp / 1000, tz=timezone.utc)
        except ValueError:
            bad_timestamp_count += 1
            continue

        if timestamp in timestamps:
            duplicate_timestamp_count += 1
        else:
            timestamps.add(timestamp)

        if previous_timestamp is not None and timestamp < previous_timestamp:
            is_time_ordered = False

        previous_timestamp = timestamp

        if min_timestamp is None or timestamp_dt < min_timestamp:
            min_timestamp = timestamp_dt

        if max_timestamp is None or timestamp_dt > max_timestamp:
            max_timestamp = timestamp_dt

        for col_index in number_columns:
            try:
                float(row[col_index])
            except ValueError:
                bad_number_count += 1


validation_report = {
    "file": str(csv_path),
    "row_count": row_count,
    "bad_column_count": bad_column_count,
    "bad_timestamp_count": bad_timestamp_count,
    "bad_number_count": bad_number_count,
    "duplicate_timestamp_count": duplicate_timestamp_count,
    "is_time_ordered": is_time_ordered,
    "min_timestamp": min_timestamp,
    "max_timestamp": max_timestamp,
}

validation_report


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest
from validators.binance_csv import validate_binance_csv


provider = BinanceProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

download_result = provider.download(
    request=request,
    output_dir=Path("data/raw/binance"),
)

validation_result = validate_binance_csv(
    csv_path=download_result.files[0],
)

download_result, validation_result


In [ ]:
from transformers.binance_ohlcv import transform_binance_csv_to_ohlcv


ohlcv_df = transform_binance_csv_to_ohlcv(
    csv_path=download_result.files[0],
)

ohlcv_df.head()


In [ ]:
from pathlib import Path

from writers.ohlcv_parquet import write_ohlcv_to_parquet


parquet_path = write_ohlcv_to_parquet(
    df=ohlcv_df,
    output_path=Path("data/bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet"),
)

parquet_path


In [ ]:
import pandas as pd


parquet_df = pd.read_parquet(parquet_path)

parquet_df.head()


In [ ]:
from pathlib import Path

from metadata.manifest import build_manifest, write_manifest


manifest = build_manifest(
    provider="binance",
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
    source_url=download_result.url,
    raw_file_path=download_result.files[0],
    parquet_file_path=parquet_path,
    validation_result=validation_result,
    ohlcv_df=ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=Path("data/bronze/binance/btcusd/2024/01/_MANIFEST.json"),
)

manifest_path


In [ ]:
import json

with open(manifest_path, "r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

loaded_manifest


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker


In [ ]:
from pathlib import Path

from metadata.success_marker import write_success_marker


bronze_dir = Path("data/bronze/binance/btcusd/2024/01")

success_path = write_success_marker(
    output_path=bronze_dir / "_SUCCESS",
)

success_path


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name


In [ ]:
from pathlib import Path


parquet_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\BTCUSDT-1m-2024-01.parquet"
)

parquet_path.exists()


In [ ]:
from uploaders.azure_blob import upload_file


parquet_blob_name = "bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=parquet_path,
    blob_name=parquet_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=parquet_blob_name,
)

blob_client.exists()



In [ ]:
manifest_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\_MANIFEST.json"
)

manifest_blob_name = "bronze/binance/btcusd/2024/01/_MANIFEST.json"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=manifest_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=manifest_blob_name,
)

blob_client.exists()


In [ ]:
success_path = Path(
    r"D:\Egyetem\On_lab_V2\data\bronze\binance\btcusd\2024\01\_SUCCESS"
)

success_blob_name = "bronze/binance/btcusd/2024/01/_SUCCESS"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=success_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=success_blob_name,
)

blob_client.exists()


In [ ]:
raw_csv_path = Path(
    r"D:\Egyetem\On_lab_V2\data\raw\binance\BTCUSDT-1m-2024-01.csv"
)

raw_csv_blob_name = "raw/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.csv"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_csv_path,
    blob_name=raw_csv_blob_name,
)


In [ ]:
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=raw_csv_blob_name,
)

blob_client.exists()


In [ ]:
import importlib
import uploaders.azure_blob

importlib.reload(uploaders.azure_blob)


In [ ]:
from uploaders.azure_blob import blob_exists


success_blob_name = "bronze/binance/btcusd/2024/01/_SUCCESS"

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=success_blob_name,
)


In [ ]:
from datetime import date

from planner.time_windows import generate_monthly_windows, generate_daily_windows


monthly_windows = generate_monthly_windows(
    start_date=date(2024, 1, 1),
    end_date=date(2024, 3, 31),
)

daily_windows = generate_daily_windows(
    start_date=date(2024, 1, 1),
    end_date=date(2024, 1, 5),
)

monthly_windows, daily_windows


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

broker_strategy_df, broker_asset_matrix_df, download_period_df


In [ ]:
from planner.download_plan import build_download_plan


plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

len(plan), plan[:3]


In [ ]:
from paths.data_paths import build_data_paths


item = plan[0]

paths = build_data_paths(
    item=item,
    interval="1m",
)

paths


In [ ]:
from pathlib import Path

from providers.binance import BinanceProvider, BinanceRequest
from validators.binance_csv import validate_binance_csv
from transformers.binance_ohlcv import transform_binance_csv_to_ohlcv
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker
from uploaders.azure_blob import upload_file, blob_exists
from paths.data_paths import build_data_paths


In [ ]:
item = plan[0]

interval = "1m"

paths = build_data_paths(
    item=item,
    interval=interval,
)

paths


In [ ]:
already_done = blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=paths.success_blob_name,
)

already_done


In [ ]:
provider = BinanceProvider()

request = BinanceRequest(
    symbol=item.broker_symbol,
    interval=interval,
    year=item.window.start_date.year,
    month=item.window.start_date.month,
)

download_result = provider.download(
    request=request,
    output_dir=paths.local_raw_file.parent,
)

download_result


In [ ]:
validation_result = validate_binance_csv(
    csv_path=download_result.files[0],
)

validation_result


In [ ]:
ohlcv_df = transform_binance_csv_to_ohlcv(
    csv_path=download_result.files[0],
)

ohlcv_df.head()


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name

In [ ]:
manifest = build_manifest(
    provider=item.broker,
    symbol=item.broker_symbol,
    interval=interval,
    year=item.window.start_date.year,
    month=item.window.start_date.month,
    source_url=download_result.url,
    raw_file_path=download_result.files[0],
    parquet_file_path=parquet_path,
    validation_result=validation_result,
    ohlcv_df=ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=paths.local_manifest_file,
)

manifest_path


In [ ]:
success_path = write_success_marker(
    output_path=paths.local_success_file,
)

success_path


In [ ]:
success_path.exists(), success_path.stat().st_size


In [ ]:
upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=download_result.files[0],
    blob_name=paths.raw_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_bronze_parquet_file,
    blob_name=paths.bronze_parquet_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_manifest_file,
    blob_name=paths.manifest_blob_name,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_success_file,
    blob_name=paths.success_blob_name,
)


In [ ]:
blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=paths.success_blob_name,
)


In [ ]:
from pipelines.binance_pipeline import run_binance_plan_item


result = run_binance_plan_item(
    item=plan[0],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

result


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)
from planner.download_plan import build_download_plan
from uploaders.azure_blob import init_azure_client


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

len(plan), container_name


In [ ]:
result = run_binance_plan_item(
    item=plan[1],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

result


In [ ]:
results = []

for item in plan[:3]:
    result = run_binance_plan_item(
        item=item,
        interval="1m",
        blob_service_client=blob_service_client,
        container_name=container_name,
    )

    results.append(result)

results


In [ ]:
from pipelines.binance_pipeline import run_binance_plan


results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results


In [ ]:
import importlib
import pipelines.binance_pipeline

importlib.reload(pipelines.binance_pipeline)

from pipelines.binance_pipeline import run_binance_plan


In [ ]:
results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results


In [ ]:
results = run_binance_plan(
    plan=plan[:3],
    interval="1m",
    blob_service_client=blob_service_client,
    container_name=container_name,
)

results

In [ ]:
import importlib
import pipelines.binance_runner

importlib.reload(pipelines.binance_runner)

from pipelines.binance_runner import run_binance_from_config


In [ ]:
results = run_binance_from_config(
    start_index=4,
    limit=1,
)

results


## Dukascopy elkezdése

In [ ]:
from pathlib import Path


output_dir = Path("data/staging/dukascopy/eurusd/2024/01")
output_dir.mkdir(parents=True, exist_ok=True)

bi5_path = output_dir / "EURUSD-2024-01-02-12_ticks.bi5"

bi5_path.write_bytes(response.content)

bi5_path


In [ ]:
bi5_path.exists(), bi5_path.stat().st_size


In [ ]:
import lzma
import struct


In [ ]:
import lzma


compressed = bi5_path.read_bytes()

decompressed = lzma.decompress(compressed)

len(compressed), len(decompressed)


In [ ]:
record_size = 20
record_count = len(decompressed) // record_size

len(decompressed), record_count, len(decompressed) % record_size


In [ ]:
import struct
from datetime import timedelta


price_scale = 100000
record_size = 20

records = []

for i in range(5):
    offset = i * record_size

    time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
        ">iii ff",
        decompressed[offset : offset + record_size],
    )

    tick_time = ts + timedelta(milliseconds=time_delta_ms)

    records.append(
        {
            "timestamp": tick_time,
            "bid": bid / price_scale,
            "ask": ask / price_scale,
            "bid_volume": bid_volume,
            "ask_volume": ask_volume,
        }
    )

records


In [ ]:
import pandas as pd
import struct
from datetime import timedelta


price_scale = 100000
record_size = 20

rows = []

for offset in range(0, len(decompressed), record_size):
    time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
        ">iii ff",
        decompressed[offset : offset + record_size],
    )

    rows.append(
        {
            "timestamp": ts + timedelta(milliseconds=time_delta_ms),
            "bid": bid / price_scale,
            "ask": ask / price_scale,
            "bid_volume": bid_volume,
            "ask_volume": ask_volume,
        }
    )

ticks_df = pd.DataFrame(rows)

ticks_df.head()


In [ ]:
ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max()


In [ ]:
ticks_df["mid"] = (ticks_df["bid"] + ticks_df["ask"]) / 2
ticks_df["volume"] = ticks_df["bid_volume"] + ticks_df["ask_volume"]

ohlcv_1m = (
    ticks_df
    .set_index("timestamp")
    .resample("1min")
    .agg(
        open=("mid", "first"),
        high=("mid", "max"),
        low=("mid", "min"),
        close=("mid", "last"),
        volume=("volume", "sum"),
    )
    .dropna()
    .reset_index()
)

ohlcv_1m.head()


In [ ]:
ohlcv_1m.shape, ohlcv_1m["timestamp"].min(), ohlcv_1m["timestamp"].max()


In [ ]:
import lzma
import struct
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd


def decode_dukascopy_bi5(
    bi5_path: Path,
    hour_start: datetime,
    price_scale: int,
) -> pd.DataFrame:
    compressed = bi5_path.read_bytes()
    decompressed = lzma.decompress(compressed)

    record_size = 20

    if len(decompressed) % record_size != 0:
        raise ValueError("Invalid Dukascopy BI5 file size.")

    rows = []

    for offset in range(0, len(decompressed), record_size):
        time_delta_ms, ask, bid, ask_volume, bid_volume = struct.unpack(
            ">iii ff",
            decompressed[offset : offset + record_size],
        )

        rows.append(
            {
                "timestamp": hour_start + timedelta(milliseconds=time_delta_ms),
                "bid": bid / price_scale,
                "ask": ask / price_scale,
                "bid_volume": bid_volume,
                "ask_volume": ask_volume,
            }
        )

    return pd.DataFrame(rows)


In [ ]:
decoded_df = decode_dukascopy_bi5(
    bi5_path=bi5_path,
    hour_start=ts,
    price_scale=100000,
)

decoded_df.head()


In [ ]:
decoded_df.shape, decoded_df["timestamp"].min(), decoded_df["timestamp"].max()


In [ ]:
from transformers.dukascopy_ticks import decode_dukascopy_bi5


decoded_df = decode_dukascopy_bi5(
    bi5_path=bi5_path,
    hour_start=ts,
    price_scale=100000,
)

decoded_df.shape, decoded_df["timestamp"].min(), decoded_df["timestamp"].max()


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from providers.dukascopy import DukascopyProvider, DukascopyRequest


provider = DukascopyProvider()

request = DukascopyRequest(
    symbol="EURUSD",
    hour_start=datetime(2024, 1, 2, 12, tzinfo=timezone.utc),
)

result = provider.download(
    request=request,
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

result


In [ ]:
result.file.exists(), result.file.stat().st_size, result.is_empty


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

from providers.dukascopy import DukascopyProvider, DukascopyRequest


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)

output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )

        download_results.append(result)

    except FileNotFoundError as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
                "type": "not_found",
            }
        )

    except RuntimeError as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
                "type": "runtime",
            }
        )

len(download_results), len(errors), errors[:5]


In [ ]:
non_empty_results = [
    result
    for result in download_results
    if not result.is_empty
]

empty_results = [
    result
    for result in download_results
    if result.is_empty
]

len(non_empty_results), len(empty_results)


In [ ]:
[(result.file.name, result.file.stat().st_size) for result in download_results[:5]]


In [ ]:
import pandas as pd

from transformers.dukascopy_ticks import decode_dukascopy_bi5


daily_tick_frames = []

for result in download_results:
    if result.is_empty:
        continue

    hour_start_text = result.file.name.replace("EURUSD-", "").replace("_ticks.bi5", "")
    hour_start = datetime.strptime(hour_start_text, "%Y-%m-%d-%H").replace(
        tzinfo=timezone.utc
    )

    tick_df = decode_dukascopy_bi5(
        bi5_path=result.file,
        hour_start=hour_start,
        price_scale=100000,
    )

    daily_tick_frames.append(tick_df)

daily_ticks_df = pd.concat(
    daily_tick_frames,
    ignore_index=True,
).sort_values("timestamp").reset_index(drop=True)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import providers.dukascopy

importlib.reload(providers.dukascopy)

from providers.dukascopy import DukascopyProvider, DukascopyRequest


In [ ]:
from datetime import datetime, timezone
from pathlib import Path


provider = DukascopyProvider()

request = DukascopyRequest(
    symbol="EURUSD",
    hour_start=datetime(2024, 1, 2, 12, tzinfo=timezone.utc),
)

test_result = provider.download(
    request=request,
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

test_result


In [ ]:
test_result.hour_start, test_result.file.exists(), test_result.file.stat().st_size, test_result.is_empty


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd

from providers.dukascopy import DukascopyProvider, DukascopyRequest
from transformers.dukascopy_ticks import decode_dukascopy_bi5


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)

output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )
        download_results.append(result)

    except Exception as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
            }
        )

tick_frames = []

for result in download_results:
    if result.is_empty:
        continue

    tick_df = decode_dukascopy_bi5(
        bi5_path=result.file,
        hour_start=result.hour_start,
        price_scale=100000,
    )

    tick_frames.append(tick_df)

daily_ticks_df = (
    pd.concat(tick_frames, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max(), len(errors)


In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd

from providers.dukascopy import DukascopyProvider, DukascopyRequest
from transformers.dukascopy_ticks import decode_dukascopy_bi5


provider = DukascopyProvider()

symbol = "EURUSD"
asset = "eurusd"
price_scale = 100000

day_start = datetime(2024, 1, 2, 0, tzinfo=timezone.utc)
output_dir = Path("data/staging/dukascopy") / asset / "2024" / "01"

download_results = []
errors = []

for hour in range(24):
    hour_start = day_start + timedelta(hours=hour)

    request = DukascopyRequest(
        symbol=symbol,
        hour_start=hour_start,
    )

    try:
        result = provider.download(
            request=request,
            output_dir=output_dir,
        )
        download_results.append(result)

    except Exception as error:
        errors.append(
            {
                "hour_start": hour_start,
                "error": str(error),
            }
        )

len(download_results), len(errors), errors[:3]


In [ ]:
[(r.hour_start, r.file.stat().st_size, r.is_empty) for r in download_results[:5]]


In [ ]:
tick_frames = []
decode_errors = []

for result in download_results:
    if result.is_empty:
        continue

    try:
        tick_df = decode_dukascopy_bi5(
            bi5_path=result.file,
            hour_start=result.hour_start,
            price_scale=price_scale,
        )

        if not tick_df.empty:
            tick_frames.append(tick_df)

    except Exception as error:
        decode_errors.append(
            {
                "hour_start": result.hour_start,
                "file": str(result.file),
                "error": str(error),
            }
        )

len(tick_frames), len(decode_errors), decode_errors[:3]


In [ ]:
daily_ticks_df = (
    pd.concat(tick_frames, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import transformers.dukascopy_ticks

importlib.reload(transformers.dukascopy_ticks)

from transformers.dukascopy_ticks import decode_dukascopy_downloads


daily_ticks_df = decode_dukascopy_downloads(
    downloads=download_results,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
daily_ticks_df.head()



In [ ]:
from pathlib import Path


raw_tick_parquet_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"
)

raw_tick_parquet_path.parent.mkdir(parents=True, exist_ok=True)

daily_ticks_df.to_parquet(
    raw_tick_parquet_path,
    index=False,
    engine="pyarrow",
)

raw_tick_parquet_path


In [ ]:
raw_tick_parquet_path.exists(), raw_tick_parquet_path.stat().st_size


In [ ]:
from uploaders.azure_blob import upload_file


raw_tick_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_tick_parquet_path,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from pathlib import Path

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_name


In [ ]:
from uploaders.azure_blob import upload_file


raw_tick_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=raw_tick_parquet_path,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from uploaders.azure_blob import blob_exists


blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_tick_blob_name,
)


In [ ]:
from writers.dukascopy_raw_parquet import write_dukascopy_raw_ticks_to_parquet


In [ ]:
raw_tick_parquet_path = write_dukascopy_raw_ticks_to_parquet(
    df=daily_ticks_df,
    output_path=Path("data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"),
)

raw_tick_parquet_path


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import download_dukascopy_day


In [ ]:
from datetime import datetime, timezone
from pathlib import Path


downloads, errors = download_dukascopy_day(
    symbol="EURUSD",
    day_start=datetime(2024, 1, 2, 0, tzinfo=timezone.utc),
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
)

len(downloads), len(errors)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
)


In [ ]:
daily_ticks_df = build_dukascopy_day_raw_ticks(
    downloads=downloads,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
    write_and_upload_dukascopy_raw_ticks,
)


In [ ]:
from pathlib import Path


local_raw_path = Path("data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet")
raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=daily_ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

raw_path


In [ ]:
from uploaders.azure_blob import blob_exists


blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)


In [ ]:
from pipelines.dukascopy_pipeline import (
    download_dukascopy_day,
    build_dukascopy_day_raw_ticks,
)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from pipelines.dukascopy_pipeline import download_dukascopy_day


downloads, errors = download_dukascopy_day(
    symbol="EURUSD",
    day_start=datetime(2024, 1, 2, 0, tzinfo=timezone.utc),
    output_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    timeout_sec=10,
    max_attempts=2,
    retry_sleep_sec=2,
)

len(downloads), len(errors), errors[:3]


In [ ]:
from pipelines.dukascopy_pipeline import build_dukascopy_day_raw_ticks


daily_ticks_df = build_dukascopy_day_raw_ticks(
    downloads=downloads,
    price_scale=100000,
)

daily_ticks_df.shape, daily_ticks_df["timestamp"].min(), daily_ticks_df["timestamp"].max()


In [ ]:
from pathlib import Path

from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks
from uploaders.azure_blob import init_azure_client, blob_exists


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

local_raw_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"
)

raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=daily_ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


In [ ]:
from datetime import date
from pathlib import Path

from planner.time_windows import TimeWindow
from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


test_window = TimeWindow(
    start_date=date(2024, 1, 2),
    end_date=date(2024, 1, 3),
)

ticks_df, errors = build_dukascopy_raw_ticks_for_window(
    symbol="EURUSD",
    window=test_window,
    staging_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    price_scale=100000,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max(), len(errors)


In [ ]:
from uploaders.azure_blob import init_azure_client, blob_exists
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

local_raw_path = Path(
    "data/raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02_to_2024-01-03.parquet"
)

raw_blob_name = "raw/dukascopy/eurusd/2024/01/EURUSD-ticks-2024-01-02_to_2024-01-03.parquet"

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=local_raw_path,
    raw_blob_name=raw_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=raw_blob_name,
)


In [ ]:
import importlib
import paths.data_paths

importlib.reload(paths.data_paths)

from paths.data_paths import build_dukascopy_raw_tick_paths


dukascopy_paths = build_dukascopy_raw_tick_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    year=2024,
    month=1,
)

dukascopy_paths


In [ ]:
from paths.data_paths import build_dukascopy_raw_tick_paths
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks


dukascopy_paths = build_dukascopy_raw_tick_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    year=2024,
    month=1,
)

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=dukascopy_paths.local_raw_tick_file,
    raw_blob_name=dukascopy_paths.raw_tick_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

raw_path


In [ ]:
blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_paths.raw_tick_blob_name,
)


In [ ]:
import importlib
import transformers.dukascopy_ohlcv

importlib.reload(transformers.dukascopy_ohlcv)

from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv


In [ ]:
dukascopy_ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

dukascopy_ohlcv_df.head()


In [ ]:
dukascopy_ohlcv_df.shape, dukascopy_ohlcv_df["timestamp"].min(), dukascopy_ohlcv_df["timestamp"].max()


In [ ]:
import importlib
import paths.data_paths

importlib.reload(paths.data_paths)

from paths.data_paths import build_dukascopy_bronze_paths


bronze_paths = build_dukascopy_bronze_paths(
    broker="dukascopy",
    asset="EURUSD",
    broker_symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
)

bronze_paths


In [ ]:
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from uploaders.azure_blob import upload_file, blob_exists


bronze_path = write_ohlcv_to_parquet(
    df=dukascopy_ohlcv_df,
    output_path=bronze_paths.local_bronze_parquet_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=bronze_path,
    blob_name=bronze_paths.bronze_parquet_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.bronze_parquet_blob_name,
)


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


manifest = build_manifest(
    provider="dukascopy",
    symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_paths.local_raw_tick_file,
    parquet_file_path=bronze_paths.local_bronze_parquet_file,
    validation_result=type(
        "ValidationResult",
        (),
        {
            "row_count": len(ticks_df),
            "bad_column_count": 0,
            "bad_timestamp_count": 0,
            "bad_number_count": 0,
            "duplicate_timestamp_count": int(ticks_df["timestamp"].duplicated().sum()),
            "is_time_ordered": bool(ticks_df["timestamp"].is_monotonic_increasing),
            "min_timestamp": ticks_df["timestamp"].min(),
            "max_timestamp": ticks_df["timestamp"].max(),
        },
    )(),
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.manifest_blob_name,
)


In [ ]:
import importlib
import validators.dukascopy_ticks

importlib.reload(validators.dukascopy_ticks)

from validators.dukascopy_ticks import validate_dukascopy_ticks


dukascopy_validation_result = validate_dukascopy_ticks(ticks_df)

dukascopy_validation_result


In [ ]:
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


manifest = build_manifest(
    provider="dukascopy",
    symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_paths.local_raw_tick_file,
    parquet_file_path=bronze_paths.local_bronze_parquet_file,
    validation_result=dukascopy_validation_result,
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.manifest_blob_name,
)


In [ ]:
from metadata.success_marker import write_success_marker
from uploaders.azure_blob import upload_file, blob_exists


success_path = write_success_marker(
    output_path=bronze_paths.local_success_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=bronze_paths.success_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=bronze_paths.success_blob_name,
)


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_strategy,
    load_broker_asset_matrix,
    load_download_period,
)
from planner.download_plan import build_download_plan


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
)

len(plan)


In [ ]:
dukascopy_plan = [
    item
    for item in plan
    if item.broker == "dukascopy"
]

len(dukascopy_plan), dukascopy_plan[:5]


In [ ]:
broker_strategy_df


In [ ]:
eurusd_item = next(
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
)

eurusd_item


In [ ]:
from pathlib import Path

from pipelines.dukascopy_pipeline import build_dukascopy_raw_ticks_for_window


ticks_df, errors = build_dukascopy_raw_ticks_for_window(
    symbol=eurusd_item.broker_symbol,
    window=eurusd_item.window,
    staging_dir=Path("data/staging/dukascopy/eurusd/2024/01"),
    price_scale=100000,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)


In [ ]:
ticks_df.shape, ticks_df["timestamp"].min(), ticks_df["timestamp"].max(), len(errors)


In [ ]:
from paths.data_paths import build_dukascopy_raw_tick_paths
from pipelines.dukascopy_pipeline import write_and_upload_dukascopy_raw_ticks
from uploaders.azure_blob import blob_exists


dukascopy_raw_paths = build_dukascopy_raw_tick_paths(
    broker=eurusd_item.broker,
    asset=eurusd_item.asset,
    broker_symbol=eurusd_item.broker_symbol,
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
)

raw_path = write_and_upload_dukascopy_raw_ticks(
    ticks_df=ticks_df,
    local_raw_path=dukascopy_raw_paths.local_raw_tick_file,
    raw_blob_name=dukascopy_raw_paths.raw_tick_blob_name,
    blob_service_client=blob_service_client,
    container_name=container_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_raw_paths.raw_tick_blob_name,
)


In [ ]:
from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv
from writers.ohlcv_parquet import write_ohlcv_to_parquet
from paths.data_paths import build_dukascopy_bronze_paths
from uploaders.azure_blob import upload_file, blob_exists


dukascopy_ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

dukascopy_bronze_paths = build_dukascopy_bronze_paths(
    broker=eurusd_item.broker,
    asset=eurusd_item.asset,
    broker_symbol=eurusd_item.broker_symbol,
    interval="1m",
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
)

bronze_path = write_ohlcv_to_parquet(
    df=dukascopy_ohlcv_df,
    output_path=dukascopy_bronze_paths.local_bronze_parquet_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=bronze_path,
    blob_name=dukascopy_bronze_paths.bronze_parquet_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.bronze_parquet_blob_name,
)


In [ ]:
from validators.dukascopy_ticks import validate_dukascopy_ticks
from metadata.manifest import build_manifest, write_manifest
from uploaders.azure_blob import upload_file, blob_exists


dukascopy_validation_result = validate_dukascopy_ticks(ticks_df)

manifest = build_manifest(
    provider=eurusd_item.broker,
    symbol=eurusd_item.broker_symbol,
    interval="1m",
    year=eurusd_item.window.start_date.year,
    month=eurusd_item.window.start_date.month,
    source_url="multiple Dukascopy hourly .bi5 files",
    raw_file_path=dukascopy_raw_paths.local_raw_tick_file,
    parquet_file_path=dukascopy_bronze_paths.local_bronze_parquet_file,
    validation_result=dukascopy_validation_result,
    ohlcv_df=dukascopy_ohlcv_df,
)

manifest_path = write_manifest(
    manifest=manifest,
    output_path=dukascopy_bronze_paths.local_manifest_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=manifest_path,
    blob_name=dukascopy_bronze_paths.manifest_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.manifest_blob_name,
)


In [ ]:
from metadata.success_marker import write_success_marker


success_path = write_success_marker(
    output_path=dukascopy_bronze_paths.local_success_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=success_path,
    blob_name=dukascopy_bronze_paths.success_blob_name,
)

blob_exists(
    blob_service_client=blob_service_client,
    container_name=container_name,
    blob_name=dukascopy_bronze_paths.success_blob_name,
)


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan_item


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_item,
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
eurusd_items = [
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
]

eurusd_items[:3]


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_items[1],
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan_item


In [ ]:
result = run_dukascopy_plan_item(
    item=eurusd_items[1],
    interval="1m",
    price_scale=100000,
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

result


In [ ]:
result


In [ ]:
import importlib
import pipelines.dukascopy_pipeline

importlib.reload(pipelines.dukascopy_pipeline)

from pipelines.dukascopy_pipeline import run_dukascopy_plan


In [ ]:
eurusd_items = [
    item
    for item in dukascopy_plan
    if item.asset == "EURUSD"
]

results = run_dukascopy_plan(
    plan=eurusd_items[:2],
    interval="1m",
    price_scale_by_asset={
        "EURUSD": 100000,
    },
    blob_service_client=blob_service_client,
    container_name=container_name,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
import importlib
import config_loader.csv_config

importlib.reload(config_loader.csv_config)


In [ ]:
from pathlib import Path

from config_loader.csv_config import load_broker_asset_settings


broker_asset_settings_df = load_broker_asset_settings(Path("config"))

broker_asset_settings_df


In [ ]:
dukascopy_price_scale_by_asset = {
    row["asset"]: int(row["price_scale"])
    for _, row in broker_asset_settings_df[
        broker_asset_settings_df["broker"] == "dukascopy"
    ].iterrows()
}

dukascopy_price_scale_by_asset


In [ ]:
import importlib
import pipelines.dukascopy_runner

importlib.reload(pipelines.dukascopy_runner)

from pipelines.dukascopy_runner import run_dukascopy_from_config


In [ ]:
results = run_dukascopy_from_config(
    start_index=2,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
results = run_dukascopy_from_config(
    start_index=8,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
results = run_dukascopy_from_config(
    start_index=14,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


In [ ]:
from pipelines.dukascopy_runner import run_dukascopy_from_config


# 2024-01 Dukascopy itemek:
# 0 = XAUUSD
# 1 = XAGUSD
# 2 = EURUSD
# 3 = US500
# 4 = DAX
# 5 = BTCUSD

results = run_dukascopy_from_config(
    start_index=5,
    limit=1,
    interval="1m",
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

results


# Végleges letöltők

### Dukascopy

In [ ]:
from pipelines.dukascopy_runner import run_dukascopy_from_config

results = run_dukascopy_from_config(
    interval="1m",
    print_progress=True,
    timeout_sec=10,
    max_attempts=2,
    retry_sleep_sec=2,
)

results

### Binance

In [ ]:
from pipelines.binance_runner import run_binance_from_config

test_results = run_binance_from_config(
    interval="1m",
    start_index=0,
    limit=1,
    max_attempts=2,
    retry_sleep_sec=2,
)

test_results


In [ ]:
from pipelines.binance_runner import run_binance_from_config

test_results = run_binance_from_config(
    interval="1m",
    start_index=0,
    limit=1,
    max_attempts=2,
    retry_sleep_sec=2,
)

test_results


In [ ]:
print("ok")


In [ ]:
from pathlib import Path

from pipelines.binance_pipeline import download_binance_with_retry
from providers.binance import BinanceRequest


class AlwaysFailProvider:
    def __init__(self):
        self.calls = 0

    def download(self, request, output_dir):
        self.calls += 1
        raise RuntimeError("fake error")


provider = AlwaysFailProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result, error = download_binance_with_retry(
    provider=provider,
    request=request,
    output_dir=Path("data/test"),
    max_attempts=2,
    retry_sleep_sec=0,
)

print(result)
print(error)
print(provider.calls)


In [ ]:
print("kernel ok")



In [ ]:
import sys
print(sys.executable)


In [ ]:
%pip install -r requirements.txt

In [ ]:
import azure.storage.blob
import pandas
import pyarrow
import pipelines.binance_pipeline
import pipelines.dukascopy_pipeline

print("onlab kernel ok")


In [ ]:
import sys
print(sys.executable)


In [ ]:
import importlib
import pipelines.binance_pipeline

importlib.reload(pipelines.binance_pipeline)

from pathlib import Path
from pipelines.binance_pipeline import download_binance_with_retry
from providers.binance import BinanceRequest


class AlwaysFailProvider:
    def __init__(self):
        self.calls = 0

    def download(self, request, output_dir):
        self.calls += 1
        raise RuntimeError("fake error")


provider = AlwaysFailProvider()

request = BinanceRequest(
    symbol="BTCUSDT",
    interval="1m",
    year=2024,
    month=1,
)

result, error = download_binance_with_retry(
    provider=provider,
    request=request,
    output_dir=Path("data/test"),
    max_attempts=2,
    retry_sleep_sec=0,
)

print(result)
print(error)
print(provider.calls)


In [ ]:
import pandas as pd

from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv


ticks_df = pd.DataFrame(
    {
        "timestamp": pd.to_datetime(
            [
                "2024-01-01 00:00:00",
                "2024-01-01 00:00:30",
                "2024-01-01 00:01:00",
                "2024-01-01 00:01:30",
                "2024-01-01 00:02:00",
            ],
            utc=True,
        ),
        "bid": [100, 101, 102, 103, 104],
        "ask": [102, 103, 104, 105, 106],
        "bid_volume": [1, 1, 1, 1, 1],
        "ask_volume": [2, 2, 2, 2, 2],
    }
)

ohlcv_1min = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="1min",
)

ohlcv_2min = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval="2min",
)

print(len(ohlcv_1min))
print(len(ohlcv_2min))
print(ohlcv_1min[["timestamp", "open", "close", "volume"]])
print(ohlcv_2min[["timestamp", "open", "close", "volume"]])


In [ ]:
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone

from metadata.manifest import build_manifest


class FakeValidationResult:
    row_count = 0
    bad_column_count = 0
    bad_timestamp_count = 0
    bad_number_count = 0
    duplicate_timestamp_count = 0
    is_time_ordered = True
    min_timestamp = None
    max_timestamp = None


empty_ohlcv_df = pd.DataFrame(
    columns=["timestamp", "open", "high", "low", "close", "volume"]
)

manifest = build_manifest(
    provider="test",
    symbol="TEST",
    interval="1m",
    year=2024,
    month=1,
    source_url="test-url",
    raw_file_path=Path("raw.csv"),
    parquet_file_path=Path("bronze.parquet"),
    validation_result=FakeValidationResult(),
    ohlcv_df=empty_ohlcv_df,
    ingestion_errors=[
        {
            "hour_start": datetime(2024, 1, 1, tzinfo=timezone.utc),
            "error": "fake error",
        }
    ],
)

manifest["data"], manifest["ingestion"]


In [ ]:
import importlib
import metadata.manifest

importlib.reload(metadata.manifest)

from metadata.manifest import build_manifest


In [ ]:
import importlib

import metadata.manifest
import pipelines.binance_pipeline
import pipelines.binance_runner
import pipelines.dukascopy_pipeline
import pipelines.dukascopy_runner

importlib.reload(metadata.manifest)
importlib.reload(pipelines.binance_pipeline)
importlib.reload(pipelines.binance_runner)
importlib.reload(pipelines.dukascopy_pipeline)
importlib.reload(pipelines.dukascopy_runner)

print("pipeline modules ok")


In [ ]:
import importlib

import planner.download_plan
import pipelines.binance_runner

importlib.reload(planner.download_plan)
importlib.reload(pipelines.binance_runner)

print("binance month range code loaded")


In [ ]:
from pipelines.binance_runner import _build_month_date_range

print(_build_month_date_range(None, None))
print(_build_month_date_range("2024-01", "2024-03"))


In [ ]:
_build_month_date_range("2026-05", "2026-05")


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_asset_matrix,
    load_broker_strategy,
    load_download_period,
)
from planner.download_plan import build_download_plan
from pipelines.binance_runner import _build_month_date_range


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

start_date, end_date = _build_month_date_range("2024-01", "2024-03")

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
    start_date=start_date,
    end_date=end_date,
)

binance_plan = [
    item
    for item in plan
    if item.broker == "binance"
]

[
    (item.broker, item.asset, item.broker_symbol, item.window.start_date, item.window.end_date)
    for item in binance_plan
]


In [ ]:
from pipelines.binance_runner import run_binance_from_config

results = run_binance_from_config(
    start_month="2024-01",
    end_month="2024-03",
    interval="1m",
)


In [ ]:
import pandas as pd

pd.DataFrame([result.__dict__ for result in results])


In [ ]:
import importlib

import pipelines.dukascopy_runner

importlib.reload(pipelines.dukascopy_runner)

print("dukascopy month range code loaded")


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_asset_matrix,
    load_broker_strategy,
    load_download_period,
)
from planner.download_plan import build_download_plan
from pipelines.binance_runner import _build_month_date_range


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

start_date, end_date = _build_month_date_range("2024-01", "2024-01")

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
    start_date=start_date,
    end_date=end_date,
)

dukascopy_plan = [
    item
    for item in plan
    if item.broker == "dukascopy"
]

[
    (index, item.broker, item.asset, item.broker_symbol, item.window.start_date, item.window.end_date)
    for index, item in enumerate(dukascopy_plan)
]


## Saxo bank

In [ ]:
import requests


SAXO_ACCESS_TOKEN = "fake-token-for-testing"
SAXO_BASE_URL = "https://gateway.saxobank.com/sim/openapi"

headers = {
    "Authorization": f"Bearer {SAXO_ACCESS_TOKEN}",
}

print("token length:", len(SAXO_ACCESS_TOKEN))
print("base url:", SAXO_BASE_URL)


In [ ]:
params = {
    "AssetTypes": "FxSpot",
    "Keywords": "EURUSD",
    "IncludeNonTradable": "true",
}

response = requests.get(
    f"{SAXO_BASE_URL}/ref/v1/instruments",
    headers=headers,
    params=params,
    timeout=30,
)

print(response.status_code)
print(response.text[:1000])


In [ ]:
eurusd_uic = 21
eurusd_asset_type = "FxSpot"

chart_params = {
    "AssetType": eurusd_asset_type,
    "Uic": eurusd_uic,
    "Horizon": 1,
    "Count": 10,
}

chart_response = requests.get(
    f"{SAXO_BASE_URL}/chart/v3/charts",
    headers=headers,
    params=chart_params,
    timeout=30,
)

print(chart_response.status_code)
print(chart_response.text[:2000])



In [ ]:
import pandas as pd

chart_data = chart_response.json()
samples = chart_data.get("Data", [])

df = pd.DataFrame(samples)
df


In [ ]:
ohlcv = pd.DataFrame()

ohlcv["timestamp"] = pd.to_datetime(
    df["Time"],
    utc=True,
    errors="coerce",
)

ohlcv["open"] = (df["OpenAsk"] + df["OpenBid"]) / 2
ohlcv["high"] = (df["HighAsk"] + df["HighBid"]) / 2
ohlcv["low"] = (df["LowAsk"] + df["LowBid"]) / 2
ohlcv["close"] = (df["CloseAsk"] + df["CloseBid"]) / 2

# Saxo FX chart mintában nincs volumen.
ohlcv["volume"] = 0.0

ohlcv


In [ ]:
for keyword in ["XAUUSD", "XAGUSD"]:
    params = {
        "AssetTypes": "FxSpot",
        "Keywords": keyword,
        "IncludeNonTradable": "true",
    }

    response = requests.get(
        f"{SAXO_BASE_URL}/ref/v1/instruments",
        headers=headers,
        params=params,
        timeout=30,
    )

    print(keyword, response.status_code)
    print(response.text[:1000])
    print()


In [ ]:
import requests
import pandas as pd


def search_saxo_instruments(
    keyword: str,
    asset_types: str = "FxSpot",
) -> pd.DataFrame:
    params = {
        "AssetTypes": asset_types,
        "Keywords": keyword,
        "IncludeNonTradable": "true",
    }

    response = requests.get(
        f"{SAXO_BASE_URL}/ref/v1/instruments",
        headers=headers,
        params=params,
        timeout=30,
    )

    print(keyword, response.status_code)

    if response.status_code != 200:
        print(response.text[:2000])
        return pd.DataFrame()

    data = response.json()
    return pd.DataFrame(data.get("Data", []))


xauusd_search = search_saxo_instruments("XAUUSD", "FxSpot")
xagusd_search = search_saxo_instruments("XAGUSD", "FxSpot")

display(xauusd_search)
display(xagusd_search)


In [ ]:
saxo_instruments = [
    {"asset": "EURUSD", "symbol": "EURUSD", "uic": 21, "asset_type": "FxSpot"},
    {"asset": "XAUUSD", "symbol": "XAUUSD", "uic": 8176, "asset_type": "FxSpot"},
    {"asset": "XAGUSD", "symbol": "XAGUSD", "uic": 8177, "asset_type": "FxSpot"},
]

for instrument in saxo_instruments:
    chart_params = {
        "AssetType": instrument["asset_type"],
        "Uic": instrument["uic"],
        "Horizon": 1,
        "Count": 10,
    }

    chart_response = requests.get(
        f"{SAXO_BASE_URL}/chart/v3/charts",
        headers=headers,
        params=chart_params,
        timeout=30,
    )

    print(instrument["asset"], chart_response.status_code)

    if chart_response.status_code == 200:
        chart_data = chart_response.json()
        samples = chart_data.get("Data", [])
        print("samples:", len(samples))
        if samples:
            print(samples[0])
    else:
        print(chart_response.text[:1000])

    print()


In [ ]:
import importlib

import config_loader.csv_config
importlib.reload(config_loader.csv_config)

from pathlib import Path
from config_loader.csv_config import load_saxo_bank_instruments

saxo_instruments_df = load_saxo_bank_instruments(Path("config"))
saxo_instruments_df


In [ ]:
import importlib
import providers.saxo_bank

importlib.reload(providers.saxo_bank)

from datetime import datetime, timezone
from providers.saxo_bank import SaxoBankProvider, SaxoChartRequest


provider = SaxoBankProvider(
    access_token=SAXO_ACCESS_TOKEN,
    base_url=SAXO_BASE_URL,
    timeout_sec=30,
)

request = SaxoChartRequest(
    uic=21,
    asset_type="FxSpot",
    horizon=1,
    start_time=datetime(2024, 1, 2, 0, 0, tzinfo=timezone.utc),
    count=10,
)

result = provider.download_chart(request)
len(result.data), result.data[:2]


In [ ]:
import importlib
import planner.saxo_bank_half_day_windows

importlib.reload(planner.saxo_bank_half_day_windows)

from datetime import date
from planner.time_windows import TimeWindow
from planner.saxo_bank_half_day_windows import generate_saxo_bank_half_day_windows

windows = generate_saxo_bank_half_day_windows(
    TimeWindow(
        start_date=date(2024, 1, 1),
        end_date=date(2024, 1, 2),
    )
)

[(w.start_time, w.end_time, w.count) for w in windows]


In [ ]:
import importlib
import transformers.saxo_bank_ohlcv

importlib.reload(transformers.saxo_bank_ohlcv)

from transformers.saxo_bank_ohlcv import transform_saxo_bank_chart_to_ohlcv

ohlcv = transform_saxo_bank_chart_to_ohlcv(result.data)
ohlcv


In [ ]:
from datetime import date
from planner.time_windows import TimeWindow
from planner.saxo_bank_half_day_windows import generate_saxo_bank_half_day_windows
from providers.saxo_bank import SaxoChartRequest
from transformers.saxo_bank_ohlcv import transform_saxo_bank_chart_to_ohlcv


test_window = TimeWindow(
    start_date=date(2024, 1, 2),
    end_date=date(2024, 1, 2),
)

half_day_windows = generate_saxo_bank_half_day_windows(test_window)

all_rows = []

for half_day_window in half_day_windows:
    request = SaxoChartRequest(
        uic=21,
        asset_type="FxSpot",
        horizon=1,
        start_time=half_day_window.start_time,
        count=half_day_window.count,
    )

    result = provider.download_chart(request)
    print(
        half_day_window.start_time,
        half_day_window.end_time,
        len(result.data),
    )

    all_rows.extend(result.data)

ohlcv = transform_saxo_bank_chart_to_ohlcv(all_rows)

ohlcv = ohlcv[
    (ohlcv["timestamp"] >= pd.Timestamp(test_window.start_date, tz="UTC")) &
    (ohlcv["timestamp"] < pd.Timestamp(date(2024, 1, 3), tz="UTC"))
].reset_index(drop=True)

len(ohlcv), ohlcv.head(), ohlcv.tail()


In [ ]:
import importlib
import paths.data_paths

importlib.reload(paths.data_paths)

from paths.data_paths import build_saxo_bank_paths

saxo_paths = build_saxo_bank_paths(
    broker="saxo_bank",
    asset="EURUSD",
    broker_symbol="EURUSD",
    interval="1m",
    year=2024,
    month=1,
)

saxo_paths


In [ ]:
import importlib
import writers.saxo_bank_raw_parquet

importlib.reload(writers.saxo_bank_raw_parquet)

from pathlib import Path
from writers.saxo_bank_raw_parquet import write_saxo_bank_raw_chart_to_parquet

test_raw_path = Path("data/test/saxo_bank/EURUSD-chart-test.parquet")

written_path = write_saxo_bank_raw_chart_to_parquet(
    chart_rows=all_rows,
    output_path=test_raw_path,
)

written_path


In [ ]:
pd.read_parquet(written_path).head()


In [ ]:
import importlib
import validators.saxo_bank_chart

importlib.reload(validators.saxo_bank_chart)

from validators.saxo_bank_chart import validate_saxo_bank_ohlcv

validation_result = validate_saxo_bank_ohlcv(ohlcv)
validation_result


In [ ]:
import importlib
import pipelines.saxo_bank_pipeline

importlib.reload(pipelines.saxo_bank_pipeline)

print("saxo bank pipeline import ok")


In [ ]:
import importlib
import pipelines.saxo_bank_runner

importlib.reload(pipelines.saxo_bank_runner)

print("saxo bank runner import ok")


In [ ]:
from pathlib import Path

from config_loader.csv_config import (
    load_broker_asset_matrix,
    load_broker_strategy,
    load_download_period,
)
from planner.download_plan import build_download_plan
from pipelines.binance_runner import _build_month_date_range


config_dir = Path("config")

broker_strategy_df = load_broker_strategy(config_dir)
broker_asset_matrix_df = load_broker_asset_matrix(config_dir)
download_period_df = load_download_period(config_dir)

start_date, end_date = _build_month_date_range("2024-01", "2024-01")

plan = build_download_plan(
    broker_strategy_df=broker_strategy_df,
    broker_asset_matrix_df=broker_asset_matrix_df,
    download_period_df=download_period_df,
    start_date=start_date,
    end_date=end_date,
)

saxo_plan = [
    item
    for item in plan
    if item.broker == "saxo_bank"
]

[
    (index, item.broker, item.asset, item.broker_symbol, item.window.start_date, item.window.end_date)
    for index, item in enumerate(saxo_plan)
]


In [ ]:
from pipelines.saxo_bank_runner import run_saxo_bank_from_config


results = run_saxo_bank_from_config(
    access_token=SAXO_ACCESS_TOKEN,
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=2,
    limit=1,
    print_progress=True,
)

results


In [ ]:
results


In [ ]:
import pandas as pd

pd.DataFrame([result.__dict__ for result in results])


In [ ]:
# XAUUSD
results = run_saxo_bank_from_config(
    access_token=SAXO_ACCESS_TOKEN,
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=0,
    limit=1,
    print_progress=True,
)

results


In [ ]:
# XAGUSD
results = run_saxo_bank_from_config(
    access_token=SAXO_ACCESS_TOKEN,
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=1,
    limit=1,
    print_progress=True,
)

results


In [ ]:
import importlib
import pipelines.monthly_ingestion_runner

importlib.reload(pipelines.monthly_ingestion_runner)

from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config

print("monthly runner with saxo ok")


In [ ]:
import sys

for module_name in [
    "pipelines.monthly_ingestion_runner",
    "pipelines.dukascopy_pipeline",
    "pipelines.dukascopy_runner",
    "pipelines.saxo_bank_pipeline",
    "pipelines.saxo_bank_runner",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

import pipelines.monthly_ingestion_runner

from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config

print("monthly runner with saxo ok")


In [ ]:
results_df = run_monthly_ingestion_from_config(
    start_month="2024-01",
    end_month="2024-01",
    brokers=["saxo_bank"],
    assets=["EURUSD"],
    interval="1m",
    saxo_access_token=SAXO_ACCESS_TOKEN,
    saxo_print_progress=True,
)

results_df


In [ ]:
results_df = run_monthly_ingestion_from_config(
    start_month="2024-01",
    end_month="2024-02",
    brokers=["saxo_bank"],
    assets=None,
    interval="1m",
    saxo_access_token=SAXO_ACCESS_TOKEN,
    saxo_base_url=SAXO_BASE_URL,
    saxo_print_progress=True,
)

results_df


In [ ]:
results_df["status"].value_counts()


In [ ]:
results_df[["broker", "asset", "year", "month", "status", "message"]]


In [ ]:
import sys

for module_name in [
    "pipelines.saxo_bank_pipeline",
    "pipelines.saxo_bank_runner",
    "pipelines.monthly_ingestion_runner",
]:
    sys.modules.pop(module_name, None)

from pipelines.saxo_bank_pipeline import run_saxo_bank_plan_item, run_saxo_bank_plan
from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config

print("saxo pipeline ok")


In [ ]:
import socket

HOST = "host.docker.internal"
PORT = 7497

with socket.create_connection((HOST, PORT), timeout=5):
    print("IB port reachable")


In [ ]:
try:
    import ib_insync
    print("ib_insync ok")
except ModuleNotFoundError as error:
    print("ib_insync missing")


In [ ]:
from ib_insync import IB

ib = IB()

ib.connect(
    host="host.docker.internal",
    port=7497,
    clientId=1,
    readonly=True,
    timeout=10,
)

print("connected:", ib.isConnected())


In [ ]:
from ib_insync import util

util.startLoop()
print("ib_insync loop ok")


In [ ]:
from ib_insync import IB

ib = IB()

ib.connect(
    host="host.docker.internal",
    port=7497,
    clientId=2,
    readonly=True,
    timeout=10,
)

print("connected:", ib.isConnected())


In [ ]:
ib.disconnect()



In [ ]:
accounts = ib.managedAccounts()
accounts


In [ ]:
from ib_insync import Forex

contract = Forex("EURUSD")
details = ib.reqContractDetails(contract)

len(details), details[:3]


In [ ]:
for detail in details:
    c = detail.contract
    print(c.symbol, c.secType, c.exchange, c.currency, c.conId, c.localSymbol)


In [ ]:
bars = ib.reqHistoricalData(
    contract,
    endDateTime="20240103 00:00:00 UTC",
    durationStr="1 D",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

len(bars), bars[:3]


In [ ]:
from ib_insync import util

df = util.df(bars)
df.head(), df.tail(), df.shape


In [ ]:
bars = ib.reqHistoricalData(
    contract,
    endDateTime="20240104 00:00:00 UTC",
    durationStr="2 D",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

df = util.df(bars)
df.head(), df.tail(), df.shape


In [ ]:
bars = ib.reqHistoricalData(
    contract,
    endDateTime="20240108 00:00:00 UTC",
    durationStr="1 W",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

df = util.df(bars)
df.head(), df.tail(), df.shape


In [ ]:
bars = ib.reqHistoricalData(
    contract,
    endDateTime="20240201 00:00:00 UTC",
    durationStr="1 M",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

df = util.df(bars)
df.head(), df.tail(), df.shape


In [ ]:
type(bars), len(bars), bars


In [ ]:
bars = ib.reqHistoricalData(
    contract,
    endDateTime="20240201 00:00:00 UTC",
    durationStr="2 W",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

print(type(bars), len(bars))

df = util.df(bars)
if df is not None:
    display(df.head())
    display(df.tail())
    print(df.shape)


In [ ]:
from ib_insync import IB, util

util.startLoop()

try:
    ib
except NameError:
    ib = IB()

if not ib.isConnected():
    ib.connect(
        host="host.docker.internal",
        port=7497,
        clientId=120,
        readonly=True,
        timeout=30,
    )

print("connected:", ib.isConnected())


In [ ]:
from ib_insync import Contract

contracts = {
    "XAUUSD_CFD": Contract(
        conId=457068913,
        symbol="XAUUSD",
        secType="CFD",
        exchange="SMART",
        currency="USD",
    ),
    "XAGUSD_CFD": Contract(
        conId=457068916,
        symbol="XAGUSD",
        secType="CFD",
        exchange="SMART",
        currency="USD",
    ),
    "US500_CFD": Contract(
        conId=111767871,
        symbol="IBUS500",
        secType="CFD",
        exchange="SMART",
        currency="USD",
    ),
    "DAX_INDEX": Contract(
        conId=825711,
        symbol="DAX",
        secType="IND",
        exchange="EUREX",
        currency="EUR",
    ),
}

contract_calendar_details = {}

for name, contract in contracts.items():
    details = ib.reqContractDetails(contract)

    print("\n===", name, "===")
    print("matches:", len(details))

    if not details:
        continue

    detail = details[0]
    contract_calendar_details[name] = detail

    print("timeZoneId:", detail.timeZoneId)
    print("tradingHours:", detail.tradingHours)
    print("liquidHours:", detail.liquidHours)


In [ ]:
from ib_insync import IB
import inspect

schedule_methods = [
    name
    for name in dir(IB)
    if "schedule" in name.lower()
]

schedule_methods


In [ ]:
for name in ["reqHistoricalSchedule", "reqHistoricalScheduleAsync"]:
    obj = getattr(IB, name)
    print(name)
    print(inspect.signature(obj))
    print(inspect.getdoc(obj))
    print("---")


In [ ]:
from ib_insync import Contract

dax_contract = Contract(
    conId=825711,
    symbol="DAX",
    secType="IND",
    exchange="EUREX",
    currency="EUR",
)

dax_schedule = ib.reqHistoricalSchedule(
    contract=dax_contract,
    numDays=31,
    endDateTime="20240201 00:00:00 UTC",
    useRTH=False,
)

dax_schedule


In [ ]:
type(dax_schedule), dax_schedule


In [ ]:
dax_schedule.sessions[:5]


In [ ]:
import pandas as pd

dax_schedule_df = pd.DataFrame(
    [
        {
            "start": session.startDateTime,
            "end": session.endDateTime,
            "ref_date": session.refDate,
        }
        for session in dax_schedule.sessions
    ]
)

dax_schedule_df


In [ ]:
dax_schedule_df["ref_date"] = pd.to_datetime(dax_schedule_df["ref_date"]).dt.date

dax_schedule_jan = dax_schedule_df[
    (dax_schedule_df["ref_date"] >= pd.to_datetime("2024-01-01").date())
    & (dax_schedule_df["ref_date"] <= pd.to_datetime("2024-01-31").date())
]

dax_schedule_jan


In [ ]:
from paths.calendar_paths import build_silver_calendar_paths

paths = build_silver_calendar_paths(
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
)

paths


In [ ]:
from pipelines.calendar_24_7_runner import run_24_7_calendar_from_config

results = run_24_7_calendar_from_config(
    asset="BTCUSD",
    start_month="2024-01",
    end_month="2024-01",
)

results


In [ ]:
from pathlib import Path
import json
import pandas as pd

calendar_path = Path("data/silver/calendar/btcusd/2024/01/BTCUSD-calendar-1m-2024-01.parquet")
manifest_path = Path("data/silver/calendar/btcusd/2024/01/_MANIFEST.json")

calendar_df = pd.read_parquet(calendar_path)

with open(manifest_path, "r", encoding="utf-8") as file:
    manifest = json.load(file)

calendar_df.head(), calendar_df.tail(), calendar_df.shape, manifest


In [ ]:
from ib_insync import Contract, util

eurusd_ib_contract = Contract(
    conId=12087792,
    symbol="EUR",
    secType="CASH",
    exchange="IDEALPRO",
    currency="USD",
)

eurusd_bars = ib.reqHistoricalData(
    eurusd_ib_contract,
    endDateTime="20240201 00:00:00 UTC",
    durationStr="2 W",
    barSizeSetting="1 min",
    whatToShow="MIDPOINT",
    useRTH=False,
    formatDate=2,
)

eurusd_df = util.df(eurusd_bars)

print("rows:", 0 if eurusd_df is None else len(eurusd_df))

if eurusd_df is not None:
    display(eurusd_df.head())
    display(eurusd_df.tail())
    print(eurusd_df.shape)


In [ ]:
from ib_insync import util
from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config

import pandas as pd

util.startLoop()

results_df = run_monthly_ingestion_from_config(
    start_month="2024-01",
    end_month="2025-12",
    brokers=["interactive_brokers"],
    assets=["EURUSD"],
    interval="1m",
    interactive_brokers_print_progress=True,
)

results_df


In [ ]:
import importlib
import config_loader.csv_config

importlib.reload(config_loader.csv_config)

from config_loader.csv_config import load_interactive_brokers_calendar_instruments

print("loader ok")


In [ ]:
from datetime import datetime, timezone

from ib_insync import util

from providers.interactive_brokers import InteractiveBrokersProvider
from providers.interactive_brokers_schedule import (
    InteractiveBrokersScheduleRequest,
    download_interactive_brokers_schedule,
)

util.startLoop()

provider = InteractiveBrokersProvider(
    host="host.docker.internal",
    port=7497,
    client_id=131,
    readonly=True,
    timeout=30,
)

request = InteractiveBrokersScheduleRequest(
    con_id=825711,
    symbol="DAX",
    security_type="IND",
    exchange="EUREX",
    currency="EUR",
    end_datetime=datetime(2024, 2, 1, 0, 0, tzinfo=timezone.utc),
    num_days=31,
    use_regular_trading_hours=False,
)

schedule_result = download_interactive_brokers_schedule(
    provider=provider,
    request=request,
)

provider.disconnect()

len(schedule_result.sessions), schedule_result.sessions[:5]


In [ ]:

from ib_insync import util

util.startLoop()


In [ ]:
import pandas as pd

schedule_df = pd.DataFrame(
    [
        {
            "start_datetime": session.start_datetime,
            "end_datetime": session.end_datetime,
            "ref_date": session.ref_date,
        }
        for session in schedule_result.sessions
    ]
)

schedule_df["ref_date"] = pd.to_datetime(schedule_df["ref_date"]).dt.date

schedule_jan_df = schedule_df[
    (schedule_df["ref_date"] >= pd.to_datetime("2024-01-01").date())
    & (schedule_df["ref_date"] <= pd.to_datetime("2024-01-31").date())
].reset_index(drop=True)

schedule_jan_df.head(), schedule_jan_df.tail(), schedule_jan_df.shape


In [ ]:
dax_calendar_df = build_minute_calendar_from_schedule_df(
    schedule_df=schedule_jan_df,
    asset="DAX",
    year=2024,
    month=1,
    timezone_name="Europe/Berlin",
)

dax_calendar_df.head(), dax_calendar_df.tail(), dax_calendar_df.shape


In [ ]:
import pandas as pd


def build_minute_calendar_from_schedule_df(
    *,
    schedule_df: pd.DataFrame,
    asset: str,
    year: int,
    month: int,
    interval: str = "1m",
    timezone_name: str = "Europe/Berlin",
) -> pd.DataFrame:
    frames = []

    pandas_freq = "1min" if interval == "1m" else interval

    for _, row in schedule_df.iterrows():
        session_start = pd.to_datetime(
            row["start_datetime"],
            format="%Y%m%d-%H:%M:%S",
        ).tz_localize(timezone_name).tz_convert("UTC")

        session_end = pd.to_datetime(
            row["end_datetime"],
            format="%Y%m%d-%H:%M:%S",
        ).tz_localize(timezone_name).tz_convert("UTC")

        timestamps = pd.date_range(
            start=session_start,
            end=session_end,
            freq=pandas_freq,
            inclusive="left",
        )

        frames.append(
            pd.DataFrame(
                {
                    "timestamp": timestamps,
                    "asset": asset.upper(),
                    "year": year,
                    "month": month,
                    "interval": interval,
                    "expected": True,
                    "calendar_source": "interactive_brokers_historical_schedule",
                    "calendar_timezone": timezone_name,
                    "session_ref_date": row["ref_date"],
                }
            )
        )

    if not frames:
        return pd.DataFrame(
            columns=[
                "timestamp",
                "asset",
                "year",
                "month",
                "interval",
                "expected",
                "calendar_source",
                "calendar_timezone",
                "session_ref_date",
            ]
        )

    calendar_df = pd.concat(frames, ignore_index=True)

    month_start = pd.Timestamp(f"{year}-{month:02d}-01", tz="UTC")
    next_month_start = month_start + pd.offsets.MonthBegin(1)

    calendar_df = calendar_df[
        (calendar_df["timestamp"] >= month_start)
        & (calendar_df["timestamp"] < next_month_start)
    ]

    return (
        calendar_df
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


In [ ]:
from pathlib import Path

ib_dax_path = Path("data/bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parquet")

ib_dax_df = pd.read_parquet(ib_dax_path)
ib_dax_df["timestamp"] = pd.to_datetime(ib_dax_df["timestamp"], utc=True)

coverage_df = dax_calendar_df.merge(
    ib_dax_df[["timestamp"]],
    on="timestamp",
    how="left",
    indicator=True,
)

coverage_df["source_row_present"] = coverage_df["_merge"] == "both"

coverage_summary = {
    "calendar_rows": len(coverage_df),
    "present_rows": int(coverage_df["source_row_present"].sum()),
    "missing_rows": int((~coverage_df["source_row_present"]).sum()),
    "coverage_ratio": coverage_df["source_row_present"].mean(),
}

coverage_summary


In [ ]:
missing_df = coverage_df[
    ~coverage_df["source_row_present"]
].copy()

missing_df.head(20), missing_df.tail(20), missing_df["timestamp"].dt.date.value_counts().sort_index()


In [ ]:
import importlib
import planner.interactive_brokers_two_week_windows

importlib.reload(planner.interactive_brokers_two_week_windows)

from datetime import date
from planner.time_windows import TimeWindow
from planner.interactive_brokers_two_week_windows import (
    generate_interactive_brokers_two_week_windows,
)

window = TimeWindow(
    start_date=date(2024, 1, 1),
    end_date=date(2024, 1, 31),
)

windows = generate_interactive_brokers_two_week_windows(window)

[
    (item.end_datetime, item.duration)
    for item in windows
]


In [ ]:
from ib_insync import util
util.startLoop()

import pandas as pd

from pipelines.calendar_interactive_brokers_runner import (
    run_interactive_brokers_calendar_from_config,
)

results = run_interactive_brokers_calendar_from_config(
    start_month="2024-01",
    end_month="2024-01",
    assets=["DAX"],
    interval="1m",
    num_days=45,
    use_regular_trading_hours=False,
    request_sleep_sec=1,
    print_progress=True,
)

pd.DataFrame([result.__dict__ for result in results])


In [ ]:
from pathlib import Path

import pandas as pd

calendar_path = Path(
    "data/silver/calendar/dax/2024/01/DAX-calendar-1m-2024-01.parquet"
)

calendar_df = pd.read_parquet(calendar_path)

calendar_df.head(), calendar_df.tail(), calendar_df.shape


In [ ]:
from pathlib import Path

import pandas as pd

calendar_path = Path("data/silver/calendar/dax/2024/01/DAX-calendar-1m-2024-01.parquet")
ib_dax_path = Path("data/bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parquet")

calendar_df = pd.read_parquet(calendar_path)
ib_dax_df = pd.read_parquet(ib_dax_path)

calendar_df["timestamp"] = pd.to_datetime(calendar_df["timestamp"], utc=True)
ib_dax_df["timestamp"] = pd.to_datetime(ib_dax_df["timestamp"], utc=True)

merged_df = calendar_df.merge(
    ib_dax_df[["timestamp"]],
    on="timestamp",
    how="left",
    indicator=True,
)

missing_df = merged_df[merged_df["_merge"] == "left_only"]

{
    "calendar_rows": len(calendar_df),
    "bronze_rows": len(ib_dax_df),
    "missing_rows": len(missing_df),
    "coverage_ratio": 1 - (len(missing_df) / len(calendar_df)),
}


In [ ]:
from pathlib import Path

from reports.data_quality_report import build_broker_ticker_overview


broker_ticker_overview_df = build_broker_ticker_overview(
    data_dir=Path("data"),
    start_month="2024-01",
    end_month="2025-12",
    brokers=None,
    tickers=None,
    interval="1m",
)

broker_ticker_overview_df


In [ ]:
from pathlib import Path

import pandas as pd


rows = []

for path in Path("data/bronze/binance/btcusd").rglob("*.parquet"):
    df = pd.read_parquet(path, columns=["timestamp"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

    rows.append(
        {
            "file": str(path),
            "dtype": str(df["timestamp"].dtype),
            "rows": len(df),
            "min_timestamp": df["timestamp"].min(),
            "max_timestamp": df["timestamp"].max(),
        }
    )

binance_debug_df = pd.DataFrame(rows).sort_values("min_timestamp")

binance_debug_df


In [ ]:
import importlib
import reports.data_quality_report

importlib.reload(reports.data_quality_report)

from pathlib import Path

from reports.data_quality_report import build_broker_ticker_overview


binance_btcusd_overview_df = build_broker_ticker_overview(
    data_dir=Path("data"),
    start_month="2025-01",
    end_month="2025-12",
    brokers=["binance"],
    tickers=["BTCUSD"],
    interval="1m",
)

binance_btcusd_overview_df


In [ ]:
import importlib.util

{
    "plotly": importlib.util.find_spec("plotly") is not None,
    "matplotlib": importlib.util.find_spec("matplotlib") is not None,
    "altair": importlib.util.find_spec("altair") is not None,
}


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


START = "2024-01-01"
END = "2024-01-31"

paths = {
    "dukascopy": Path("data/bronze/dukascopy/dax/2024/01/DEUIDXEUR-1m-2024-01.parquet"),
    "interactive_brokers": Path("data/bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parquet"),
}

dfs = []

for broker, path in paths.items():
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    df = df[
        (df["timestamp"] >= pd.Timestamp(START, tz="UTC"))
        & (df["timestamp"] <= pd.Timestamp(END, tz="UTC"))
    ].copy()

    df["broker"] = broker

    if broker == "dukascopy":
        df["close_scaled"] = df["close"] / 10
    else:
        df["close_scaled"] = df["close"]

    dfs.append(df[["timestamp", "broker", "close_scaled"]])

dax_df = pd.concat(dfs, ignore_index=True)

plt.figure(figsize=(16, 6))

for broker, broker_df in dax_df.groupby("broker"):
    plt.plot(
        broker_df["timestamp"],
        broker_df["close_scaled"],
        label=broker,
        linewidth=1,
    )

plt.title("DAX close árfolyam összehasonlítás - 2024-01")
plt.xlabel("Idő")
plt.ylabel("Close")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


DATA_DIR = Path("data")
TICKER = "DAX"
YEAR = "2024"
MONTH = "01"

paths = {
    "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "dax" / YEAR / MONTH / "DEUIDXEUR-1m-2024-01.parquet",
    "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "dax" / YEAR / MONTH / "DAX-1m-2024-01.parquet",
}

settings_df = pd.read_csv("config/broker_asset_settings.csv")

price_scale_by_broker_ticker = {
    (row["broker"], row["asset"]): row["price_scale"]
    for _, row in settings_df.iterrows()
}

frames = []

for broker, path in paths.items():
    if not path.exists():
        print(f"missing: {path}")
        continue

    df = pd.read_parquet(
        path,
        columns=["timestamp", "close"],
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
    )

    price_scale = price_scale_by_from pathlib import Path

import pandas as pd


dukascopy_path = Path("data/bronze/dukascopy/dax/2024/01/DEUIDXEUR-1m-2024-01.parquet")
ib_path = Path("data/bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parquet")

dukascopy_df = pd.read_parquet(
    dukascopy_path,
    columns=["timestamp", "close"],
)

ib_df = pd.read_parquet(
    ib_path,
    columns=["timestamp", "close"],
)

dukascopy_df["timestamp"] = pd.to_datetime(
    dukascopy_df["timestamp"],
    utc=True,
)

ib_df["timestamp"] = pd.to_datetime(
    ib_df["timestamp"],
    utc=True,
)

merged_df = dukascopy_df.merge(
    ib_df,
    on="timestamp",
    how="inner",
    suffixes=("_dukascopy", "_ib"),
)

merged_df["dukascopy_div_10"] = merged_df["close_dukascopy"] / 10
merged_df["dukascopy_div_100"] = merged_df["close_dukascopy"] / 100

merged_df[
    [
        "close_dukascopy",
        "dukascopy_div_10",
        "dukascopy_div_100",
        "close_ib",
    ]
].describe()
broker_ticker.get(
        (broker, TICKER),
        1,
    )

    df["close_scaled"] = df["close"] / price_scale
    df["broker"] = broker

    frames.append(
        df[
            [
                "timestamp",
                "broker",
                "close_scaled",
            ]
        ]
    )

dax_df = pd.concat(
    frames,
    ignore_index=True,
)

plt.figure(figsize=(16, 6))

for broker, broker_df in dax_df.groupby("broker"):
    broker_df = broker_df.sort_values("timestamp")

    plt.plot(
        broker_df["timestamp"],
        broker_df["close_scaled"],
        label=broker,
        linewidth=1,
    )

plt.title("DAX close árfolyam brókerenként - 2024-01")
plt.xlabel("Idő")
plt.ylabel("Close")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plot_df = merged_df.copy()
plot_df["dukascopy_scaled"] = plot_df["close_dukascopy"] / 10

plt.figure(figsize=(16, 6))

plt.plot(
    plot_df["timestamp"],
    plot_df["dukascopy_scaled"],
    label="dukascopy / 10",
    linewidth=1,
)

plt.plot(
    plot_df["timestamp"],
    plot_df["close_ib"],
    label="interactive_brokers",
    linewidth=1,
)

plt.title("DAX close összehasonlítás közös timestampen - 2024-01")
plt.xlabel("Idő")
plt.ylabel("Close")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plot_df["diff"] = plot_df["dukascopy_scaled"] - plot_df["close_ib"]

plt.figure(figsize=(16, 4))

plt.plot(
    plot_df["timestamp"],
    plot_df["diff"],
    linewidth=1,
)

plt.axhline(0, color="black", linewidth=1)

plt.title("DAX close különbség: Dukascopy / 10 - IB")
plt.xlabel("Idő")
plt.ylabel("Pont eltérés")
plt.grid(True)
plt.show()


In [ ]:
from pathlib import Path

import pandas as pd


DATA_DIR = Path("data")
YEAR = "2024"
MONTH = "01"

CANDIDATE_SCALES = [1, 10, 100, 1000, 100000]

FILES = {
    "BTCUSD": {
        "binance": DATA_DIR / "bronze" / "binance" / "btcusd" / YEAR / MONTH / "BTCUSDT-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "btcusd" / YEAR / MONTH / "BTCUSD-1m-2024-01.parquet",
    },
    "DAX": {
        "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "dax" / YEAR / MONTH / "DAX-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "dax" / YEAR / MONTH / "DEUIDXEUR-1m-2024-01.parquet",
    },
    "EURUSD": {
        "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "eurusd" / YEAR / MONTH / "EURUSD-1m-2024-01.parquet",
        "saxo_bank": DATA_DIR / "bronze" / "saxo_bank" / "eurusd" / YEAR / MONTH / "EURUSD-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "eurusd" / YEAR / MONTH / "EURUSD-1m-2024-01.parquet",
    },
    "XAUUSD": {
        "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "xauusd" / YEAR / MONTH / "XAUUSD-1m-2024-01.parquet",
        "saxo_bank": DATA_DIR / "bronze" / "saxo_bank" / "xauusd" / YEAR / MONTH / "XAUUSD-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "xauusd" / YEAR / MONTH / "XAUUSD-1m-2024-01.parquet",
    },
    "XAGUSD": {
        "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "xagusd" / YEAR / MONTH / "XAGUSD-1m-2024-01.parquet",
        "saxo_bank": DATA_DIR / "bronze" / "saxo_bank" / "xagusd" / YEAR / MONTH / "XAGUSD-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "xagusd" / YEAR / MONTH / "XAGUSD-1m-2024-01.parquet",
    },
    "US500": {
        "interactive_brokers": DATA_DIR / "bronze" / "interactive_brokers" / "us500" / YEAR / MONTH / "US500-1m-2024-01.parquet",
        "dukascopy": DATA_DIR / "bronze" / "dukascopy" / "us500" / YEAR / MONTH / "USA500IDXUSD-1m-2024-01.parquet",
    },
}


REFERENCE_BROKER = {
    "BTCUSD": "binance",
    "DAX": "interactive_brokers",
    "EURUSD": "interactive_brokers",
    "XAUUSD": "interactive_brokers",
    "XAGUSD": "interactive_brokers",
    "US500": "interactive_brokers",
}


def load_close(path):
    df = pd.read_parquet(path, columns=["timestamp", "close"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df


scale_records = []

for ticker, broker_paths in FILES.items():
    reference_broker = REFERENCE_BROKER[ticker]
    reference_path = broker_paths[reference_broker]

    if not reference_path.exists():
        print(f"missing reference: {ticker} {reference_broker}")
        continue

    reference_df = load_close(reference_path).rename(
        columns={"close": "reference_close"}
    )

    for broker, path in broker_paths.items():
        if broker == reference_broker:
            continue

        if not path.exists():
            print(f"missing: {ticker} {broker}")
            continue

        broker_df = load_close(path).rename(
            columns={"close": "broker_close"}
        )

        merged_df = reference_df.merge(
            broker_df,
            on="timestamp",
            how="inner",
        )

        if merged_df.empty:
            continue

        best_scale = None
        best_mean_abs_pct_diff = None
        best_mean_abs_diff = None

        for scale in CANDIDATE_SCALES:
            scaled_close = merged_df["broker_close"] / scale

            abs_diff = (scaled_close - merged_df["reference_close"]).abs()
            abs_pct_diff = abs_diff / merged_df["reference_close"].abs() * 100

            mean_abs_diff = abs_diff.mean()
            mean_abs_pct_diff = abs_pct_diff.mean()

            if (
                best_mean_abs_pct_diff is None
                or mean_abs_pct_diff < best_mean_abs_pct_diff
            ):
                best_scale = scale
                best_mean_abs_pct_diff = mean_abs_pct_diff
                best_mean_abs_diff = mean_abs_diff

        scale_records.append(
            {
                "ticker": ticker,
                "reference_broker": reference_broker,
                "broker": broker,
                "common_rows": len(merged_df),
                "best_scale": best_scale,
                "mean_abs_diff": best_mean_abs_diff,
                "mean_abs_pct_diff": best_mean_abs_pct_diff,
            }
        )

scale_check_df = pd.DataFrame(scale_records).sort_values(
    ["ticker", "broker"]
)

scale_check_df


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


binance_path = Path("data/bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet")
dukascopy_path = Path("data/bronze/dukascopy/btcusd/2024/01/BTCUSD-1m-2024-01.parquet")

binance_df = pd.read_parquet(
    binance_path,
    columns=["timestamp", "close"],
)

dukascopy_df = pd.read_parquet(
    dukascopy_path,
    columns=["timestamp", "close"],
)

binance_df["timestamp"] = pd.to_datetime(
    binance_df["timestamp"],
    utc=True,
)

dukascopy_df["timestamp"] = pd.to_datetime(
    dukascopy_df["timestamp"],
    utc=True,
)

binance_df["broker"] = "binance"
dukascopy_df["broker"] = "dukascopy"

btc_df = pd.concat(
    [
        binance_df,
        dukascopy_df,
    ],
    ignore_index=True,
)

plt.figure(figsize=(16, 6))

for broker, broker_df in btc_df.groupby("broker"):
    broker_df = broker_df.sort_values("timestamp")

    plt.plot(
        broker_df["timestamp"],
        broker_df["close"],
        label=broker,
        linewidth=1,
    )

plt.title("BTCUSD close árfolyam brókerenként - 2024-01")
plt.xlabel("Idő")
plt.ylabel("Close")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
btc_plot_df = pd.concat(
    [
        binance_df.assign(
            broker="binance",
            close_scaled=binance_df["close"],
        ),
        dukascopy_df.assign(
            broker="dukascopy x10",
            close_scaled=dukascopy_df["close"] * 10,
        ),
    ],
    ignore_index=True,
)

plt.figure(figsize=(16, 6))

for broker, broker_df in btc_plot_df.groupby("broker"):
    broker_df = broker_df.sort_values("timestamp")

    plt.plot(
        broker_df["timestamp"],
        broker_df["close_scaled"],
        label=broker,
        linewidth=1,
    )

plt.title("BTCUSD close árfolyam skálázva - 2024-01")
plt.xlabel("Idő")
plt.ylabel("Close")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from pathlib import Path

import pandas as pd

raw_path = next(Path("data/raw/dukascopy/dax/2024/01").glob("*.parquet"))

raw_df = pd.read_parquet(raw_path)

raw_path, raw_df.dtypes, raw_df.head(), raw_df.shape


In [ ]:
from pathlib import Path

import pandas as pd

from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker
from paths.data_paths import build_data_paths
from planner.download_plan import DownloadPlanItem
from planner.time_windows import TimeWindow
from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv
from uploaders.azure_blob import init_azure_client, upload_file
from validators.dukascopy_ticks import validate_dukascopy_ticks
from writers.ohlcv_parquet import write_ohlcv_to_parquet


item = DownloadPlanItem(
    broker="dukascopy",
    asset="DAX",
    broker_symbol="DEUIDXEUR",
    window_type="monthly",
    window=TimeWindow(
        start_date=pd.Timestamp("2024-01-01").date(),
        end_date=pd.Timestamp("2024-01-31").date(),
    ),
    rate_limit_sleep_sec=0,
)

interval = "1m"
price_scale = 10

paths = build_data_paths(
    item=item,
    interval=interval,
    data_dir=Path("data"),
)

blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

raw_blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=paths.raw_blob_name,
)

paths.local_raw_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(paths.local_raw_file, "wb") as file:
    file.write(raw_blob_client.download_blob().readall())

ticks_df = pd.read_parquet(paths.local_raw_file)

validation_result = validate_dukascopy_ticks(ticks_df)

ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
    ticks_df=ticks_df,
    interval=interval,
)

for column in ["open", "high", "low", "close"]:
    ohlcv_df[column] = ohlcv_df[column] / price_scale

parquet_path = write_ohlcv_to_parquet(
    df=ohlcv_df,
    output_path=paths.local_bronze_parquet_file,
)

manifest = build_manifest(
    provider=item.broker,
    symbol=item.broker_symbol,
    interval=interval,
    year=2024,
    month=1,
    source_url=paths.raw_blob_name,
    raw_file_path=paths.local_raw_file,
    parquet_file_path=parquet_path,
    validation_result=validation_result,
    ohlcv_df=ohlcv_df,
    ingestion_errors=[],
)

write_manifest(
    manifest=manifest,
    output_path=paths.local_manifest_file,
)

write_success_marker(
    output_path=paths.local_success_file,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_bronze_parquet_file,
    blob_name=paths.bronze_parquet_blob_name,
    overwrite=True,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_manifest_file,
    blob_name=paths.manifest_blob_name,
    overwrite=True,
)

upload_file(
    blob_service_client=blob_service_client,
    container_name=container_name,
    local_path=paths.local_success_file,
    blob_name=paths.success_blob_name,
    overwrite=True,
)

ohlcv_df["close"].describe()


In [ ]:
from pathlib import Path

import pandas as pd

from metadata.manifest import build_manifest, write_manifest
from metadata.success_marker import write_success_marker
from paths.data_paths import (
    build_dukascopy_bronze_paths,
    build_dukascopy_raw_tick_paths,
)
from transformers.dukascopy_ohlcv import transform_dukascopy_ticks_to_ohlcv
from uploaders.azure_blob import init_azure_client, upload_file
from validators.dukascopy_ticks import validate_dukascopy_ticks
from writers.ohlcv_parquet import write_ohlcv_to_parquet


BROKER = "dukascopy"
INTERVAL = "1m"
START_MONTH = "2024-01"
END_MONTH = "2025-12"

BROKER_SYMBOL_BY_ASSET = {
    "EURUSD": "EURUSD",
    "XAUUSD": "XAUUSD",
    "XAGUSD": "XAGUSD",
    "US500": "USA500IDXUSD",
    "DAX": "DEUIDXEUR",
    "BTCUSD": "BTCUSD",
}

PRICE_SCALE_BY_ASSET = {
    "EURUSD": 1,
    "XAUUSD": 10,
    "XAGUSD": 1,
    "US500": 10,
    "DAX": 10,
    "BTCUSD": 0.1,
}


def iter_months(
    start_month: str,
    end_month: str,
):
    start = pd.Period(start_month, freq="M")
    end = pd.Period(end_month, freq="M")

    for period in pd.period_range(start, end):
        yield period.year, period.month


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

results = []

for year, month in iter_months(
    START_MONTH,
    END_MONTH,
):
    for asset, broker_symbol in BROKER_SYMBOL_BY_ASSET.items():
        price_scale = PRICE_SCALE_BY_ASSET[asset]

        print(f"rebuilding {asset} {year}-{month:02d}")

        raw_paths = build_dukascopy_raw_tick_paths(
            broker=BROKER,
            asset=asset,
            broker_symbol=broker_symbol,
            year=year,
            month=month,
            data_dir=Path("data"),
        )

        bronze_paths = build_dukascopy_bronze_paths(
            broker=BROKER,
            asset=asset,
            broker_symbol=broker_symbol,
            interval=INTERVAL,
            year=year,
            month=month,
            data_dir=Path("data"),
        )

        raw_blob_client = blob_service_client.get_blob_client(
            container=container_name,
            blob=raw_paths.raw_tick_blob_name,
        )

        if not raw_blob_client.exists():
            print(f"  missing raw: {raw_paths.raw_tick_blob_name}")

            results.append(
                {
                    "status": "missing_raw",
                    "asset": asset,
                    "year": year,
                    "month": month,
                    "rows": 0,
                }
            )

            continue

        raw_paths.local_raw_tick_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with open(raw_paths.local_raw_tick_file, "wb") as file:
            file.write(raw_blob_client.download_blob().readall())

        ticks_df = pd.read_parquet(
            raw_paths.local_raw_tick_file,
        )

        validation_result = validate_dukascopy_ticks(
            ticks_df,
        )

        ohlcv_df = transform_dukascopy_ticks_to_ohlcv(
            ticks_df=ticks_df,
            interval=INTERVAL,
        )

        for column in ["open", "high", "low", "close"]:
            ohlcv_df[column] = ohlcv_df[column] / price_scale

        bronze_path = write_ohlcv_to_parquet(
            df=ohlcv_df,
            output_path=bronze_paths.local_bronze_parquet_file,
        )

        manifest = build_manifest(
            provider=BROKER,
            symbol=broker_symbol,
            interval=INTERVAL,
            year=year,
            month=month,
            source_url=raw_paths.raw_tick_blob_name,
            raw_file_path=raw_paths.local_raw_tick_file,
            parquet_file_path=bronze_path,
            validation_result=validation_result,
            ohlcv_df=ohlcv_df,
            ingestion_errors=[],
        )

        manifest_path = write_manifest(
            manifest=manifest,
            output_path=bronze_paths.local_manifest_file,
        )

        success_path = write_success_marker(
            output_path=bronze_paths.local_success_file,
        )

        upload_file(
            blob_service_client=blob_service_client,
            container_name=container_name,
            local_path=bronze_path,
            blob_name=bronze_paths.bronze_parquet_blob_name,
            overwrite=True,
        )

        upload_file(
            blob_service_client=blob_service_client,
            container_name=container_name,
            local_path=manifest_path,
            blob_name=bronze_paths.manifest_blob_name,
            overwrite=True,
        )

        upload_file(
            blob_service_client=blob_service_client,
            container_name=container_name,
            local_path=success_path,
            blob_name=bronze_paths.success_blob_name,
            overwrite=True,
        )

        results.append(
            {
                "status": "uploaded",
                "asset": asset,
                "year": year,
                "month": month,
                "rows": len(ohlcv_df),
                "min_timestamp": ohlcv_df["timestamp"].min(),
                "max_timestamp": ohlcv_df["timestamp"].max(),
                "price_scale": price_scale,
            }
        )

        print(f"  uploaded rows={len(ohlcv_df)}")

results_df = pd.DataFrame(results)

results_df


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


DATA_DIR = Path("data")
START_MONTH = "2024-01"
END_MONTH = "2024-01"

TICKERS = ["BTCUSD", "DAX", "EURUSD", "US500", "XAGUSD", "XAUUSD"]

FILES = {
    "BTCUSD": {
        "binance": {
            "folder": "btcusd",
            "symbol": "BTCUSDT",
        },
        "dukascopy": {
            "folder": "btcusd",
            "symbol": "BTCUSD",
        },
    },
    "DAX": {
        "dukascopy": {
            "folder": "dax",
            "symbol": "DEUIDXEUR",
        },
        "interactive_brokers": {
            "folder": "dax",
            "symbol": "DAX",
        },
    },
    "EURUSD": {
        "dukascopy": {
            "folder": "eurusd",
            "symbol": "EURUSD",
        },
        "interactive_brokers": {
            "folder": "eurusd",
            "symbol": "EURUSD",
        },
        "saxo_bank": {
            "folder": "eurusd",
            "symbol": "EURUSD",
        },
    },
    "US500": {
        "dukascopy": {
            "folder": "us500",
            "symbol": "USA500IDXUSD",
        },
        "interactive_brokers": {
            "folder": "us500",
            "symbol": "US500",
        },
    },
    "XAGUSD": {
        "dukascopy": {
            "folder": "xagusd",
            "symbol": "XAGUSD",
        },
        "interactive_brokers": {
            "folder": "xagusd",
            "symbol": "XAGUSD",
        },
        "saxo_bank": {
            "folder": "xagusd",
            "symbol": "XAGUSD",
        },
    },
    "XAUUSD": {
        "dukascopy": {
            "folder": "xauusd",
            "symbol": "XAUUSD",
        },
        "interactive_brokers": {
            "folder": "xauusd",
            "symbol": "XAUUSD",
        },
        "saxo_bank": {
            "folder": "xauusd",
            "symbol": "XAUUSD",
        },
    },
}


def iter_months(
    start_month: str,
    end_month: str,
):
    start = pd.Period(start_month, freq="M")
    end = pd.Period(end_month, freq="M")

    for period in pd.period_range(start, end):
        yield str(period.year), f"{period.month:02d}"


def load_close_series(
    *,
    broker: str,
    ticker: str,
    start_month: str,
    end_month: str,
) -> pd.DataFrame:
    config = FILES[ticker][broker]
    frames = []

    for year, month in iter_months(
        start_month,
        end_month,
    ):
        path = (
            DATA_DIR
            / "bronze"
            / broker
            / config["folder"]
            / year
            / month
            / f"{config['symbol']}-1m-{year}-{month}.parquet"
        )

        if not path.exists():
            continue

        df = pd.read_parquet(
            path,
            columns=[
                "timestamp",
                "close",
            ],
        )

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            utc=True,
        )

        df["broker"] = broker
        df["ticker"] = ticker

        frames.append(df)

    if not frames:
        return pd.DataFrame(
            columns=[
                "timestamp",
                "close",
                "broker",
                "ticker",
            ]
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


def plot_ticker_close(
    *,
    ticker: str,
    start_month: str,
    end_month: str,
):
    frames = []

    for broker in FILES[ticker]:
        broker_df = load_close_series(
            broker=broker,
            ticker=ticker,
            start_month=start_month,
            end_month=end_month,
        )

        if not broker_df.empty:
            frames.append(broker_df)

    if not frames:
        print(f"No data for {ticker}")
        return

    ticker_df = pd.concat(
        frames,
        ignore_index=True,
    )

    plt.figure(figsize=(16, 5))

    for broker, broker_df in ticker_df.groupby("broker"):
        broker_df = broker_df.sort_values("timestamp")

        plt.scatter(
            broker_df["timestamp"],
            broker_df["close"],
            label=broker,
            s=4,
            alpha=0.7,
        )

    plt.title(f"{ticker} close árfolyam brókerenként - {start_month}..{end_month}")
    plt.xlabel("Idő")
    plt.ylabel("Close")
    plt.legend()
    plt.grid(True)
    plt.show()


for ticker in TICKERS:
    plot_ticker_close(
        ticker=ticker,
        start_month=START_MONTH,
        end_month=END_MONTH,
    )


In [ ]:
import plotly.express as px


ticker_df = load_close_series(
    broker="dukascopy",
    ticker="DAX",
    start_month="2024-01",
    end_month="2024-01",
)

ib_df = load_close_series(
    broker="interactive_brokers",
    ticker="DAX",
    start_month="2024-01",
    end_month="2024-01",
)

plot_df = pd.concat(
    [ticker_df, ib_df],
    ignore_index=True,
)

fig = px.scatter(
    plot_df,
    x="timestamp",
    y="close",
    color="broker",
    title="DAX close árfolyam brókerenként - 2024-01",
    labels={
        "timestamp": "Idő",
        "close": "Close",
        "broker": "Bróker",
    },
)

fig.update_traces(
    marker={
        "size": 3,
        "opacity": 0.65,
    }
)

fig.update_layout(
    height=550,
    hovermode="x unified",
)

fig.show()


In [ ]:
import importlib
import quality.data_loader

importlib.reload(quality.data_loader)

from quality.data_loader import load_bronze_ohlcv


bronze_df = load_bronze_ohlcv(
    start_month="2024-01",
    end_month="2024-02",
    brokers=["binance"],
    assets=["BTCUSD"],
    source="azure",
    print_progress=True,
)

bronze_df.head(), bronze_df.tail(), bronze_df.shape


In [ ]:
import pandas as pd

quality_df = quality_base_df.copy()

quality_df["timestamp"] = pd.to_datetime(
    quality_df["timestamp"],
    utc=True,
    errors="coerce",
)

quality_df["timestamp_minute"] = quality_df["timestamp"].dt.floor("min")


In [ ]:
quality_df

In [ ]:
before_df = (
    quality_base_df
    .groupby(["broker", "asset"], dropna=False)
    .agg(
        original_rows=("timestamp", "count"),
        original_start=("timestamp", "min"),
        original_end=("timestamp", "max"),
    )
    .reset_index()
)

after_df = (
    quality_df
    .groupby(["broker", "asset"], dropna=False)
    .agg(
        quality_rows=("timestamp", "count"),
        quality_start=("timestamp", "min"),
        quality_end=("timestamp", "max"),
    )
    .reset_index()
)

check_df = before_df.merge(
    after_df,
    on=["broker", "asset"],
    how="outer",
)

check_df["row_diff"] = check_df["quality_rows"] - check_df["original_rows"]

check_df


In [ ]:
from quality.calendar_alignment import load_calendar


calendar_df = load_calendar(
    start_month="2024-01",
    end_month="2024-02",
    assets=None,
    interval="1m",
    source="azure",
    print_progress=True,
)

calendar_df

In [ ]:
import pandas as pd


source_df = quality_base_df.copy()
calendar_filter_df = calendar_df.copy()

source_df = source_df[source_df["broker"].notna()].copy()

source_df["timestamp"] = pd.to_datetime(
    source_df["timestamp"],
    utc=True,
    errors="coerce",
)

calendar_filter_df["timestamp"] = pd.to_datetime(
    calendar_filter_df["timestamp"],
    utc=True,
    errors="coerce",
)

source_df["time"] = source_df["timestamp"].dt.floor("min")
calendar_filter_df["time"] = calendar_filter_df["timestamp"].dt.floor("min")

calendar_columns = [
    "time",
    "asset",
]

optional_calendar_columns = [
    "year",
    "month",
    "expected",
    "calendar_source",
    "calendar_timezone",
    "session_ref_date",
]

calendar_columns += [
    column
    for column in optional_calendar_columns
    if column in calendar_filter_df.columns
]

calendar_key_df = (
    calendar_filter_df[calendar_columns]
    .dropna(subset=["time", "asset"])
    .drop_duplicates(subset=["time", "asset"])
    .copy()
)

filtered_df = source_df.merge(
    calendar_key_df[["time", "asset"]],
    on=["time", "asset"],
    how="inner",
)

minute_broker_df = (
    filtered_df
    .sort_values(["asset", "broker", "time", "timestamp"])
    .groupby(["time", "asset", "broker"], as_index=False)
    .agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume=("volume", "sum"),
        source_rows=("timestamp", "count"),
    )
)

wide_df = minute_broker_df.pivot(
    index=["time", "asset"],
    columns="broker",
    values=[
        "open",
        "high",
        "low",
        "close",
        "volume",
        "source_rows",
    ],
)

wide_df.columns = [
    f"{broker}_{field}"
    for field, broker in wide_df.columns
]

wide_df = wide_df.reset_index()

broker_close_columns = [
    column
    for column in wide_df.columns
    if column.endswith("_close")
]

wide_df["broker_count"] = wide_df[broker_close_columns].notna().sum(axis=1)

wide_df["close_min"] = wide_df[broker_close_columns].min(axis=1)
wide_df["close_max"] = wide_df[broker_close_columns].max(axis=1)

wide_df["close_diff"] = wide_df["close_max"] - wide_df["close_min"]

wide_df["close_diff_pct"] = (
    wide_df["close_diff"] / wide_df["close_min"] * 100
)

quality_wide_df = calendar_key_df.merge(
    wide_df,
    on=["time", "asset"],
    how="left",
)

quality_wide_df["generated_candle_slot"] = True

front_columns = [
    "time",
    "asset",
    "generated_candle_slot",
]

for column in [
    "year",
    "month",
    "expected",
    "calendar_source",
    "calendar_timezone",
    "session_ref_date",
    "broker_count",
    "close_diff",
    "close_diff_pct",
]:
    if column in quality_wide_df.columns:
        front_columns.append(column)

other_columns = [
    column
    for column in quality_wide_df.columns
    if column not in front_columns
]

quality_wide_df = quality_wide_df[
    front_columns + other_columns
].sort_values(
    ["asset", "time"]
).reset_index(drop=True)

quality_wide_df


In [ ]:
generated_df = quality_wide_df.copy()

open_columns = [
    column
    for column in generated_df.columns
    if column.endswith("_open")
]

high_columns = [
    column
    for column in generated_df.columns
    if column.endswith("_high")
]

low_columns = [
    column
    for column in generated_df.columns
    if column.endswith("_low")
]

close_columns = [
    column
    for column in generated_df.columns
    if column.endswith("_close")
]

generated_df["generated_open"] = generated_df[open_columns].median(
    axis=1,
    skipna=True,
)

generated_df["generated_high"] = generated_df[high_columns].median(
    axis=1,
    skipna=True,
)

generated_df["generated_low"] = generated_df[low_columns].median(
    axis=1,
    skipna=True,
)

generated_df["generated_close"] = generated_df[close_columns].median(
    axis=1,
    skipna=True,
)

generated_df["generated_volume"] = pd.NA

generated_df["consensus_quality"] = "missing"

generated_df.loc[
    generated_df["broker_count"] == 1,
    "consensus_quality",
] = "single_source"

generated_df.loc[
    generated_df["broker_count"] >= 2,
    "consensus_quality",
] = "multi_source"

generated_df[
    [
        "time",
        "asset",
        "generated_open",
        "generated_high",
        "generated_low",
        "generated_close",
        "generated_volume",
        "broker_count",
        "consensus_quality",
        "close_diff",
        "close_diff_pct",
    ]
]


In [ ]:
def generate_candles_from_brokers(
    quality_wide_df: pd.DataFrame,
    *,
    method: int = 0,
) -> pd.DataFrame:
    generated_df = quality_wide_df.copy()

    open_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_open")
    ]

    high_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_high")
    ]

    low_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_low")
    ]

    close_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_close")
    ]

    if method == 0:
        method_name = "median_ohlc"

        generated_df["generated_open"] = generated_df[open_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_high"] = generated_df[high_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_low"] = generated_df[low_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_close"] = generated_df[close_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_volume"] = pd.NA

    else:
        raise ValueError(f"Unsupported generation method: {method}")

    generated_df["generation_method_id"] = method
    generated_df["generation_method"] = method_name

    generated_df["consensus_quality"] = "missing"

    generated_df.loc[
        generated_df["broker_count"] == 1,
        "consensus_quality",
    ] = "single_source"

    generated_df.loc[
        generated_df["broker_count"] >= 2,
        "consensus_quality",
    ] = "multi_source"

    generated_df["is_generated"] = generated_df["generated_close"].notna()

    return generated_df


In [ ]:
generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=0,
)

generated_df[
    [
        "time",
        "asset",
        "generated_open",
        "generated_high",
        "generated_low",
        "generated_close",
        "generated_volume",
        "broker_count",
        "consensus_quality",
        "generation_method_id",
        "generation_method",
        "is_generated",
        "close_diff",
        "close_diff_pct",
    ]
]


In [ ]:
import plotly.graph_objects as go


btc_plot_df = generated_df[
    generated_df["asset"] == "BTCUSD"
].copy()

btc_plot_df = btc_plot_df.sort_values("time")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["binance_close"],
        mode="lines",
        name="binance close",
    )
)

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["dukascopy_close"],
        mode="lines",
        name="dukascopy close",
    )
)

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["generated_close"],
        mode="lines",
        name="generated close",
        line=dict(
            dash="dash",
            width=3,
        ),
    )
)

fig.update_layout(
    title="BTCUSD close árfolyam - Binance, Dukascopy, Generated",
    xaxis_title="Idő",
    yaxis_title="Close",
    hovermode="x unified",
    height=600,
)

fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

from quality.data_loader import load_bronze_ohlcv
from quality.calendar_alignment import load_calendar


START_MONTH = "2024-01"
END_MONTH = "2024-02"
INTERVAL = "1m"


# 1. Bronze adatok letöltése egyszer
quality_base_df = load_bronze_ohlcv(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=None,
    assets=None,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)


# 2. Calendar letöltése egyszer
calendar_df = load_calendar(
    start_month=START_MONTH,
    end_month=END_MONTH,
    assets=None,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)


# 3. Másolat készítése, az eredeti quality_base_df érintetlen marad
source_df = quality_base_df.copy()
calendar_filter_df = calendar_df.copy()


# 4. Timestamp normalizálás percre
source_df = source_df[source_df["broker"].notna()].copy()

source_df["timestamp"] = pd.to_datetime(
    source_df["timestamp"],
    utc=True,
    errors="coerce",
)

calendar_filter_df["timestamp"] = pd.to_datetime(
    calendar_filter_df["timestamp"],
    utc=True,
    errors="coerce",
)

source_df["time"] = source_df["timestamp"].dt.floor("min")
calendar_filter_df["time"] = calendar_filter_df["timestamp"].dt.floor("min")


# 5. Calendar alapján csak az elvárt gyertyahelyek megtartása
calendar_columns = [
    "time",
    "asset",
]

optional_calendar_columns = [
    "year",
    "month",
    "expected",
    "calendar_source",
    "calendar_timezone",
    "session_ref_date",
]

calendar_columns += [
    column
    for column in optional_calendar_columns
    if column in calendar_filter_df.columns
]

calendar_key_df = (
    calendar_filter_df[calendar_columns]
    .dropna(subset=["time", "asset"])
    .drop_duplicates(subset=["time", "asset"])
    .copy()
)

filtered_df = source_df.merge(
    calendar_key_df[["time", "asset"]],
    on=["time", "asset"],
    how="inner",
)


# 6. Egy broker / egy asset / egy perc szintre hozás
minute_broker_df = (
    filtered_df
    .sort_values(["asset", "broker", "time", "timestamp"])
    .groupby(["time", "asset", "broker"], as_index=False)
    .agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume=("volume", "sum"),
        source_rows=("timestamp", "count"),
    )
)


# 7. Széles tábla: broker_x OHLCV oszlopok
wide_df = minute_broker_df.pivot(
    index=["time", "asset"],
    columns="broker",
    values=[
        "open",
        "high",
        "low",
        "close",
        "volume",
        "source_rows",
    ],
)

wide_df.columns = [
    f"{broker}_{field}"
    for field, broker in wide_df.columns
]

wide_df = wide_df.reset_index()

broker_close_columns = [
    column
    for column in wide_df.columns
    if column.endswith("_close")
]

wide_df["broker_count"] = wide_df[broker_close_columns].notna().sum(axis=1)
wide_df["close_min"] = wide_df[broker_close_columns].min(axis=1)
wide_df["close_max"] = wide_df[broker_close_columns].max(axis=1)
wide_df["close_diff"] = wide_df["close_max"] - wide_df["close_min"]
wide_df["close_diff_pct"] = wide_df["close_diff"] / wide_df["close_min"] * 100

quality_wide_df = calendar_key_df.merge(
    wide_df,
    on=["time", "asset"],
    how="left",
)

quality_wide_df["generated_candle_slot"] = True


# 8. Generált gyertya több módszerrel bővíthető függvényben
def generate_candles_from_brokers(
    quality_wide_df: pd.DataFrame,
    *,
    method: int = 0,
) -> pd.DataFrame:
    generated_df = quality_wide_df.copy()

    open_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_open")
    ]

    high_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_high")
    ]

    low_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_low")
    ]

    close_columns = [
        column
        for column in generated_df.columns
        if column.endswith("_close")
    ]

    if method == 0:
        method_name = "median_ohlc"

        generated_df["generated_open"] = generated_df[open_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_high"] = generated_df[high_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_low"] = generated_df[low_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_close"] = generated_df[close_columns].median(
            axis=1,
            skipna=True,
        )
        generated_df["generated_volume"] = pd.NA

    else:
        raise ValueError(f"Unsupported generation method: {method}")

    generated_df["generation_method_id"] = method
    generated_df["generation_method"] = method_name

    generated_df["consensus_quality"] = "missing"

    generated_df.loc[
        generated_df["broker_count"] == 1,
        "consensus_quality",
    ] = "single_source"

    generated_df.loc[
        generated_df["broker_count"] >= 2,
        "consensus_quality",
    ] = "multi_source"

    generated_df["is_generated"] = generated_df["generated_close"].notna()

    return generated_df


generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=0,
)


# 9. BTCUSD: Binance + Dukascopy + Generated kirajzolása
btc_plot_df = generated_df[
    generated_df["asset"] == "BTCUSD"
].copy()

btc_plot_df = btc_plot_df.sort_values("time")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["binance_close"],
        mode="lines",
        name="binance close",
    )
)

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["dukascopy_close"],
        mode="lines",
        name="dukascopy close",
    )
)

fig.add_trace(
    go.Scatter(
        x=btc_plot_df["time"],
        y=btc_plot_df["generated_close"],
        mode="lines",
        name="generated close",
        line=dict(
            dash="dash",
            width=3,
        ),
    )
)

fig.update_layout(
    title="BTCUSD close árfolyam - Binance, Dukascopy, Generated",
    xaxis_title="Idő",
    yaxis_title="Close",
    hovermode="x unified",
    height=650,
)

fig.show()


In [ ]:
xau_plot_df = generated_df[
    generated_df["asset"] == "XAUUSD"
].copy()

xau_plot_df = xau_plot_df.sort_values("time")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=xau_plot_df["time"],
        y=xau_plot_df["dukascopy_close"],
        mode="lines",
        name="dukascopy close",
    )
)

fig.add_trace(
    go.Scatter(
        x=xau_plot_df["time"],
        y=xau_plot_df["interactive_brokers_close"],
        mode="lines",
        name="interactive_brokers close",
    )
)

fig.add_trace(
    go.Scatter(
        x=xau_plot_df["time"],
        y=xau_plot_df["saxo_bank_close"],
        mode="lines",
        name="saxo_bank close",
    )
)

fig.add_trace(
    go.Scatter(
        x=xau_plot_df["time"],
        y=xau_plot_df["generated_close"],
        mode="lines",
        name="generated close",
        line=dict(
            dash="dash",
            width=3,
        ),
    )
)

fig.update_layout(
    title="XAUUSD close árfolyam - Dukascopy, IB, Saxo, Generated",
    xaxis_title="Idő",
    yaxis_title="Close",
    hovermode="x unified",
    height=650,
)

fig.show()


In [ ]:
import importlib
import quality.candle_generation

importlib.reload(quality.candle_generation)

from quality.candle_generation import build_generated_candles

print("candle_generation import ok")


In [ ]:
generated_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    method=0,
    source="azure",
    print_progress=True,
)

generated_df.shape, generated_df.head()


In [ ]:
generated_df.groupby(
    ["asset", "consensus_quality"],
    dropna=False,
).agg(
    rows=("time", "count"),
    generated_rows=("is_generated", "sum"),
    avg_broker_count=("broker_count", "mean"),
    max_close_diff_pct=("close_diff_pct", "max"),
).reset_index()


In [ ]:
import plotly.graph_objects as go


btc_plot_df = generated_df[
    generated_df["asset"] == "BTCUSD"
].sort_values("time")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=btc_plot_df["time"],
    y=btc_plot_df["binance_close"],
    mode="lines",
    name="binance close",
))

fig.add_trace(go.Scatter(
    x=btc_plot_df["time"],
    y=btc_plot_df["dukascopy_close"],
    mode="lines",
    name="dukascopy close",
))

fig.add_trace(go.Scatter(
    x=btc_plot_df["time"],
    y=btc_plot_df["generated_close"],
    mode="lines",
    name="generated close",
    line=dict(dash="dash", width=3),
))

fig.update_layout(
    title="BTCUSD teszt - Binance, Dukascopy, Generated",
    xaxis_title="Idő",
    yaxis_title="Close",
    hovermode="x unified",
    height=600,
)

fig.show()


In [ ]:
from metadata.silver_manifest import build_silver_generated_ohlcv_manifest


test_month_df = silver_ohlcv_df[
    (silver_ohlcv_df["asset"] == "BTCUSD")
    & (silver_ohlcv_df["year"] == 2024)
    & (silver_ohlcv_df["month"] == 1)
].copy()

manifest = build_silver_generated_ohlcv_manifest(
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
    generation_method_id=0,
    generation_method="median_ohlc",
    parquet_file_path=paths.local_parquet_file,
    silver_ohlcv_df=test_month_df,
)

manifest


In [ ]:
from quality.candle_generation import build_generated_candles
from quality.silver_ohlcv import build_silver_ohlcv_from_generated


generated_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD"],
    interval="1m",
    method=0,
    source="azure",
    print_progress=True,
)

silver_ohlcv_df = build_silver_ohlcv_from_generated(
    generated_df,
)

silver_ohlcv_df.head()


In [ ]:
from pathlib import Path

import importlib

import pipelines.silver_generated_ohlcv_pipeline

importlib.reload(pipelines.silver_generated_ohlcv_pipeline)

from pipelines.silver_generated_ohlcv_pipeline import (
    write_and_upload_silver_generated_ohlcv_item,
)
from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

result = write_and_upload_silver_generated_ohlcv_item(
    silver_ohlcv_df=silver_ohlcv_df,
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
    generation_method_id=0,
    generation_method="median_ohlc",
    blob_service_client=blob_service_client,
    container_name=container_name,
    overwrite=True,
)

result


In [ ]:
from io import BytesIO

import pandas as pd


blob_name = "silver/generated_ohlcv/btcusd/2024/01/BTCUSD-generated-1m-2024-01.parquet"

container_client = blob_service_client.get_container_client(
    container_name,
)

blob_bytes = container_client.get_blob_client(
    blob_name,
).download_blob().readall()

uploaded_df = pd.read_parquet(
    BytesIO(blob_bytes),
)

uploaded_df.head(), uploaded_df.tail(), uploaded_df.shape


In [ ]:
import json


manifest_blob_name = "silver/generated_ohlcv/btcusd/2024/01/_MANIFEST.json"

manifest_bytes = container_client.get_blob_client(
    manifest_blob_name,
).download_blob().readall()

manifest = json.loads(
    manifest_bytes.decode("utf-8"),
)

manifest


In [ ]:
upload_plan_preview_df = (
    silver_ohlcv_df[
        [
            "asset",
            "year",
            "month",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "year",
            "month",
            "asset",
        ]
    )
    .reset_index(drop=True)
)

upload_plan_preview_df


In [ ]:
from io import BytesIO
from pathlib import Path

import pandas as pd

from uploaders.azure_blob import init_azure_client


ASSET = "BTCUSD"
YEAR = 2024
MONTH = 1
INTERVAL = "1m"

blob_name = (
    f"silver/generated_ohlcv/{ASSET.lower()}/{YEAR}/{MONTH:02d}/"
    f"{ASSET}-generated-{INTERVAL}-{YEAR}-{MONTH:02d}.parquet"
)

blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_client = blob_service_client.get_container_client(
    container_name,
)

blob_bytes = container_client.get_blob_client(
    blob_name,
).download_blob().readall()

silver_test_df = pd.read_parquet(
    BytesIO(blob_bytes),
)

print("blob:", blob_name)
print("shape:", silver_test_df.shape)

display(silver_test_df.head())
display(silver_test_df.tail())

display(
    silver_test_df.groupby(
        ["asset", "consensus_quality"],
        dropna=False,
    )
    .agg(
        rows=("timestamp", "count"),
        generated_rows=("is_generated", "sum"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        max_close_diff_pct=("close_diff_pct", "max"),
    )
    .reset_index()
)


In [ ]:
import importlib
import quality.silver_ohlcv

importlib.reload(quality.silver_ohlcv)

from quality.silver_ohlcv import (
    build_silver_ohlcv_from_generated,
    build_silver_ohlcv_summary,
)


silver_ohlcv_df = build_silver_ohlcv_from_generated(
    generated_df,
)

silver_ohlcv_df.groupby(
    ["asset", "candle_quality"],
    dropna=False,
).agg(
    rows=("timestamp", "count"),
    avg_close_diff_pct=("close_diff_pct", "mean"),
    max_close_diff_pct=("close_diff_pct", "max"),
).reset_index()


In [ ]:
silver_ohlcv_df[
    silver_ohlcv_df["candle_quality"] == "bad"
][
    [
        "timestamp",
        "asset",
        "open",
        "high",
        "low",
        "close",
        "consensus_quality",
        "broker_count",
        "close_diff",
        "close_diff_pct",
        "candle_quality",
    ]
]


In [ ]:
import importlib

import quality.silver_ohlcv

importlib.reload(quality.silver_ohlcv)

from quality.silver_ohlcv import (
    build_silver_ohlcv_from_generated,
    build_silver_ohlcv_summary,
)


silver_ohlcv_df = build_silver_ohlcv_from_generated(
    generated_df,
)

silver_summary_df = build_silver_ohlcv_summary(
    silver_ohlcv_df,
)

silver_summary_df


In [ ]:
silver_ohlcv_df.groupby(
    ["asset", "is_outlier"],
    dropna=False,
).agg(
    rows=("timestamp", "count"),
    max_abs_return_pct=("abs_return_pct", "max"),
).reset_index()


In [ ]:
silver_ohlcv_df[
    silver_ohlcv_df["is_outlier"]
][
    [
        "timestamp",
        "asset",
        "open",
        "high",
        "low",
        "close",
        "return_pct",
        "abs_return_pct",
        "consensus_quality",
        "candle_quality",
        "broker_count",
        "close_diff_pct",
        "is_outlier",
    ]
]


In [ ]:
import importlib
import quality.silver_ohlcv

importlib.reload(quality.silver_ohlcv)

from quality.silver_ohlcv import build_silver_ohlcv_from_generated


silver_ohlcv_df = build_silver_ohlcv_from_generated(
    generated_df,
)

silver_ohlcv_df.groupby(
    ["asset", "quality_status"],
    dropna=False,
).agg(
    rows=("timestamp", "count"),
    generated_rows=("is_generated", "sum"),
    outliers=("is_outlier", "sum"),
    max_close_diff_pct=("close_diff_pct", "max"),
    max_abs_return_pct=("abs_return_pct", "max"),
).reset_index()


In [ ]:
import importlib
import quality.broker_ranking

importlib.reload(quality.broker_ranking)

from quality.broker_ranking import build_broker_ranking


broker_ranking_df = build_broker_ranking(
    generated_df,
)

broker_ranking_df


In [ ]:
import importlib
import pipelines.silver_quality_runner

importlib.reload(pipelines.silver_quality_runner)

from pipelines.silver_quality_runner import run_silver_quality_from_config


START_MONTH = "2024-01"
END_MONTH = "2024-02"
BROKERS = None
ASSETS = None
INTERVAL = "1m"
GENERATION_METHOD = 0


quality_result = run_silver_quality_from_config(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval=INTERVAL,
    generation_method=GENERATION_METHOD,
    source="azure",
    print_progress=True,
    show_figures=True,
)

generated_df = quality_result["generated"]
silver_ohlcv_df = quality_result["silver_ohlcv"]
silver_summary_df = quality_result["silver_summary"]
broker_ranking_df = quality_result["broker_ranking"]
asset_quality_ranking_df = quality_result["asset_quality_ranking"]

silver_summary_df


In [ ]:
from pathlib import Path

Path("pipelines/silver_quality_runner.py").exists()


In [ ]:
import importlib
import quality.candle_generation

importlib.reload(quality.candle_generation)


from quality.candle_generation import build_generated_candles


generated_preferred_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    method=1,
    source="azure",
    print_progress=True,
)

generated_preferred_df[
    [
        "time",
        "asset",
        "generated_close",
        "selected_broker",
        "broker_count",
        "consensus_quality",
        "generation_method",
    ]
].head()


In [ ]:
generated_preferred_df.groupby(
    ["asset", "selected_broker"],
    dropna=False,
).agg(
    rows=("time", "count"),
    avg_broker_count=("broker_count", "mean"),
).reset_index()


In [ ]:
import importlib
import quality.candle_generation

importlib.reload(quality.candle_generation)

from quality.candle_generation import (
    build_broker_wide_table,
    _build_preferred_broker_order,
)
from quality.data_loader import load_bronze_ohlcv
from quality.calendar_alignment import load_calendar


bronze_test_df = load_bronze_ohlcv(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    source="azure",
    print_progress=True,
)

calendar_test_df = load_calendar(
    start_month="2024-01",
    end_month="2024-01",
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    source="azure",
    print_progress=True,
)

quality_wide_test_df = build_broker_wide_table(
    bronze_df=bronze_test_df,
    calendar_df=calendar_test_df,
)

preferred_order = _build_preferred_broker_order(
    quality_wide_test_df,
)

preferred_order


In [ ]:
from quality.candle_generation import build_generated_candles


generated_preferred_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    method=1,
    source="azure",
    print_progress=True,
)

generated_preferred_df.groupby(
    ["asset", "selected_broker"],
    dropna=False,
).agg(
    rows=("time", "count"),
    avg_broker_count=("broker_count", "mean"),
).reset_index()


In [ ]:
generated_median_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD", "XAUUSD"],
    interval="1m",
    method=0,
    source="azure",
    print_progress=False,
)

compare_df = generated_median_df[
    [
        "time",
        "asset",
        "generated_close",
    ]
].rename(
    columns={
        "generated_close": "median_close",
    }
).merge(
    generated_preferred_df[
        [
            "time",
            "asset",
            "generated_close",
            "selected_broker",
        ]
    ].rename(
        columns={
            "generated_close": "preferred_close",
        }
    ),
    on=[
        "time",
        "asset",
    ],
    how="inner",
)

compare_df["diff"] = (
    compare_df["preferred_close"]
    - compare_df["median_close"]
)

compare_df["diff_pct"] = (
    compare_df["diff"].abs()
    / compare_df["median_close"]
    * 100
)

compare_df.groupby("asset").agg(
    rows=("time", "count"),
    mean_diff_pct=("diff_pct", "mean"),
    median_diff_pct=("diff_pct", "median"),
    max_diff_pct=("diff_pct", "max"),
).reset_index()


In [ ]:
broker_names = [
    "dukascopy",
    "interactive_brokers",
    "saxo_bank",
]

asset_df = quality_wide_test_df[
    quality_wide_test_df["asset"] == "XAUUSD"
].copy()

close_columns = [
    f"{broker}_close"
    for broker in broker_names
    if f"{broker}_close" in asset_df.columns
]

asset_df["median_reference_close"] = asset_df[close_columns].median(
    axis=1,
    skipna=True,
)

ranking_records = []

for broker in broker_names:
    close_column = f"{broker}_close"

    broker_present_mask = asset_df[close_column].notna()
    broker_rows = int(broker_present_mask.sum())

    broker_close = asset_df[close_column]

    abs_diff_pct = (
        (broker_close - asset_df["median_reference_close"]).abs()
        / asset_df["median_reference_close"]
        * 100
    )

    ranking_records.append(
        {
            "broker": broker,
            "calendar_rows": len(asset_df),
            "broker_rows": broker_rows,
            "coverage_ratio": broker_rows / len(asset_df),
            "mean_abs_diff_pct": abs_diff_pct.mean(),
            "median_abs_diff_pct": abs_diff_pct.median(),
            "max_abs_diff_pct": abs_diff_pct.max(),
        }
    )

pd.DataFrame(ranking_records).sort_values(
    [
        "coverage_ratio",
        "mean_abs_diff_pct",
        "broker",
    ],
    ascending=[
        False,
        True,
        True,
    ],
)


In [ ]:
from paths.silver_paths import build_silver_generated_ohlcv_paths


test_paths = build_silver_generated_ohlcv_paths(
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
    generation_method_id=1,
)

test_paths


In [ ]:
from quality.candle_generation import build_generated_candles
from quality.silver_ohlcv import build_silver_ohlcv_from_generated


generated_method_1_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD"],
    interval="1m",
    method=1,
    source="azure",
    print_progress=True,
)

silver_method_1_df = build_silver_ohlcv_from_generated(
    generated_method_1_df,
)

silver_method_1_df.head(), silver_method_1_df.shape


In [ ]:
from pathlib import Path

from pipelines.silver_generated_ohlcv_pipeline import (
    write_and_upload_silver_generated_ohlcv_item,
)
from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

test_upload_result = write_and_upload_silver_generated_ohlcv_item(
    silver_ohlcv_df=silver_method_1_df,
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
    generation_method_id=1,
    generation_method="preferred_broker_ohlc",
    blob_service_client=blob_service_client,
    container_name=container_name,
    overwrite=True,
)

test_upload_result


In [ ]:
from io import BytesIO

import pandas as pd


blob_name = (
    "silver/generated_ohlcv/method_1/btcusd/2024/01/"
    "BTCUSD-generated-1m-2024-01.parquet"
)

container_client = blob_service_client.get_container_client(
    container_name,
)

blob_bytes = container_client.get_blob_client(
    blob_name,
).download_blob().readall()

uploaded_method_1_df = pd.read_parquet(
    BytesIO(blob_bytes),
)

uploaded_method_1_df.head(), uploaded_method_1_df.tail(), uploaded_method_1_df.shape


In [ ]:
from quality.candle_generation import build_generated_candles
from quality.silver_ohlcv import build_silver_ohlcv_from_generated
from pipelines.silver_generated_ohlcv_pipeline import (
    write_and_upload_silver_generated_ohlcv_item,
)


generated_method_0_df = build_generated_candles(
    start_month="2024-01",
    end_month="2024-01",
    brokers=None,
    assets=["BTCUSD"],
    interval="1m",
    method=0,
    source="azure",
    print_progress=True,
)

silver_method_0_df = build_silver_ohlcv_from_generated(
    generated_method_0_df,
)

test_upload_result_method_0 = write_and_upload_silver_generated_ohlcv_item(
    silver_ohlcv_df=silver_method_0_df,
    asset="BTCUSD",
    interval="1m",
    year=2024,
    month=1,
    generation_method_id=0,
    generation_method="median_ohlc",
    blob_service_client=blob_service_client,
    container_name=container_name,
    overwrite=True,
)

test_upload_result_method_0


In [ ]:
from pathlib import Path

import pandas as pd

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_client = blob_service_client.get_container_client(
    container_name,
)

records = []

for blob in container_client.list_blobs(
    name_starts_with="silver/generated_ohlcv/"
):
    blob_name = blob.name

    parts = blob_name.split("/")

    records.append(
        {
            "blob_name": blob_name,
            "size_mb": blob.size / 1024 / 1024,
            "method": parts[2] if len(parts) > 2 else None,
            "asset": parts[3].upper() if len(parts) > 3 else None,
            "year": parts[4] if len(parts) > 4 else None,
            "month": parts[5] if len(parts) > 5 else None,
            "file": parts[-1],
        }
    )

silver_blobs_df = pd.DataFrame(records)

silver_blobs_df.sort_values(
    [
        "method",
        "asset",
        "year",
        "month",
        "file",
    ]
)


In [ ]:
silver_blob_summary_df = (
    silver_blobs_df
    .groupby(
        [
            "method",
            "asset",
            "year",
            "month",
        ],
        dropna=False,
    )
    .agg(
        files=("file", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "method",
            "asset",
            "year",
            "month",
        ]
    )
)

silver_blob_summary_df


In [ ]:
silver_blobs_df.groupby(
    ["method", "asset"],
    dropna=False,
).agg(
    files=("file", "count"),
    months=("month", "nunique"),
    total_size_mb=("size_mb", "sum"),
).reset_index()


In [ ]:
silver_parquet_df = silver_blobs_df[
    silver_blobs_df["blob_name"].str.endswith(".parquet")
    & silver_blobs_df["blob_name"].str.contains(
        r"^silver/generated_ohlcv/method_[01]/",
        regex=True,
    )
].copy()

silver_parquet_df = silver_parquet_df[
    silver_parquet_df["asset"].notna()
].copy()

silver_parquet_df.groupby(
    ["method", "asset", "year"],
    dropna=False,
).agg(
    months=("month", "nunique"),
    parquet_files=("file", "count"),
    total_size_mb=("size_mb", "sum"),
).reset_index().sort_values(
    ["method", "asset", "year"]
)


In [ ]:
silver_parquet_df.groupby(
    ["method", "asset"],
    dropna=False,
).agg(
    years=("year", "nunique"),
    months=("month", "nunique"),
    parquet_files=("file", "count"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    total_size_mb=("size_mb", "sum"),
).reset_index().sort_values(
    ["method", "asset"]
)


In [ ]:
from io import BytesIO
from pathlib import Path

import json
import pandas as pd

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

container_client = blob_service_client.get_container_client(
    container_name,
)

METHOD = "method_0"
ASSET = "XAUUSD"
YEAR = 2025
MONTH = 12
INTERVAL = "1m"

parquet_blob_name = (
    f"silver/generated_ohlcv/{METHOD}/{ASSET.lower()}/{YEAR}/{MONTH:02d}/"
    f"{ASSET}-generated-{INTERVAL}-{YEAR}-{MONTH:02d}.parquet"
)

parquet_bytes = container_client.get_blob_client(
    parquet_blob_name,
).download_blob().readall()

silver_check_df = pd.read_parquet(
    BytesIO(parquet_bytes),
)

print("blob:", parquet_blob_name)
print("shape:", silver_check_df.shape)

display(silver_check_df.head())
display(silver_check_df.tail())

display(silver_check_df.dtypes)


In [ ]:
display(
    silver_check_df.groupby(
        [
            "asset",
            "consensus_quality",
            "candle_quality",
            "is_outlier",
            "quality_status",
        ],
        dropna=False,
    )
    .agg(
        rows=("timestamp", "count"),
        generated_rows=("is_generated", "sum"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        max_close_diff_pct=("close_diff_pct", "max"),
        max_abs_return_pct=("abs_return_pct", "max"),
    )
    .reset_index()
)


In [ ]:
manifest_blob_name = (
    f"silver/generated_ohlcv/{METHOD}/{ASSET.lower()}/{YEAR}/{MONTH:02d}/"
    "_MANIFEST.json"
)

manifest_bytes = container_client.get_blob_client(
    manifest_blob_name,
).download_blob().readall()

manifest = json.loads(
    manifest_bytes.decode("utf-8"),
)

manifest


In [ ]:
METHOD = "method_1"

parquet_blob_name = (
    f"silver/generated_ohlcv/{METHOD}/{ASSET.lower()}/{YEAR}/{MONTH:02d}/"
    f"{ASSET}-generated-{INTERVAL}-{YEAR}-{MONTH:02d}.parquet"
)

parquet_bytes = container_client.get_blob_client(
    parquet_blob_name,
).download_blob().readall()

silver_check_method_1_df = pd.read_parquet(
    BytesIO(parquet_bytes),
)

print("method_0 shape:", silver_check_df.shape)
print("method_1 shape:", silver_check_method_1_df.shape)

display(silver_check_method_1_df.head())
display(silver_check_method_1_df.tail())

display(
    silver_check_method_1_df.groupby(
        [
            "asset",
            "consensus_quality",
            "candle_quality",
            "is_outlier",
            "quality_status",
        ],
        dropna=False,
    )
    .agg(
        rows=("timestamp", "count"),
        generated_rows=("is_generated", "sum"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        max_close_diff_pct=("close_diff_pct", "max"),
        max_abs_return_pct=("abs_return_pct", "max"),
    )
    .reset_index()
)


In [ ]:
import plotly.graph_objects as go


plot_df = silver_check_df.copy()
plot_df = plot_df.sort_values("timestamp")

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["close"],
        mode="lines",
        name="silver close",
    )
)

bad_df = plot_df[
    plot_df["quality_status"] == "bad"
]

fig.add_trace(
    go.Scatter(
        x=bad_df["timestamp"],
        y=bad_df["close"],
        mode="markers",
        name="bad quality",
        marker=dict(
            size=8,
            color="red",
        ),
    )
)

warning_df = plot_df[
    plot_df["quality_status"] == "warning"
]

fig.add_trace(
    go.Scatter(
        x=warning_df["timestamp"],
        y=warning_df["close"],
        mode="markers",
        name="warning quality",
        marker=dict(
            size=5,
            color="orange",
            opacity=0.6,
        ),
    )
)

fig.update_layout(
    title="Silver generated OHLCV close - quality jelölésekkel",
    xaxis_title="Idő",
    yaxis_title="Close",
    hovermode="x unified",
    height=650,
)

fig.show()


In [ ]:
print("Hello world")

In [ ]:
from pipelines.silver_quality_runner import run_silver_quality_from_config


result = run_silver_quality_from_config(
    start_month="2024-01",
    end_month="2024-01",
    assets=None,
    generation_method=1,
    source="azure",
    print_progress=False,
    show_figures=False,
)

silver_ohlcv_df = result["silver_ohlcv"]

selected_broker_summary_df = (
    silver_ohlcv_df
    .groupby(
        [
            "asset",
            "selected_broker",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        [
            "asset",
            "rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

selected_broker_summary_df


In [ ]:
silver_ohlcv_df[
    [
        "timestamp",
        "asset",
        "generation_method",
        "selected_broker",
        "broker_count",
        "close",
        "consensus_quality",
        "quality_status",
    ]
].tail(50)


In [10]:
from pathlib import Path

import pandas as pd

from uploaders.azure_blob import init_azure_client


blob_service_client, container_name = init_azure_client(
    env_path=Path(".env"),
)

blob_name = (
    "silver/generated_ohlcv/method_1/"
    "eurusd/2024/01/"
    "EURUSD-generated-1m-2024-01.parquet"
)

local_path = Path("data/tmp/EURUSD-method-1-2024-01.parquet")
local_path.parent.mkdir(parents=True, exist_ok=True)

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name,
)

with open(local_path, "wb") as file:
    file.write(blob_client.download_blob().readall())

df = pd.read_parquet(local_path)

df[
    [
        "timestamp",
        "asset",
        "generation_method_id",
        "generation_method",
        "selected_broker",
        "broker_count",
        "close",
    ]
].head(20)


,timestamp,asset,generation_method_id,generation_method,selected_broker,broker_count,close
0,2024-01-01 19:00:00+00:00,EURUSD,1,preferred_broker_ohlc,NaN,NaN,NaN
1,2024-01-01 19:01:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103675
2,2024-01-01 19:02:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103725
3,2024-01-01 19:03:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103730
4,2024-01-01 19:04:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103735
5,2024-01-01 19:05:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103730
6,2024-01-01 19:06:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103745
7,2024-01-01 19:07:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103800
8,2024-01-01 19:08:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103730
9,2024-01-01 19:09:00+00:00,EURUSD,1,preferred_broker_ohlc,saxo_bank,1.0,1.103750


In [11]:
(
    df
    .groupby(
        [
            "asset",
            "selected_broker",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)


,asset,selected_broker,rows
2,EURUSD,saxo_bank,30639
3,EURUSD,NaN,783
0,EURUSD,dukascopy,221
1,EURUSD,interactive_brokers,7
